# Notebook 04 — STAFFIM Inspection Comment Analysis for VHS Interpretation

### IRIS Auto Fraud Decision Platform — BNA Assurances Tunisie

| Field | Value |
|-------|-------|
| **Run ID** | `VHS_BALANCED_V2_20260630_133318` |
| **Profile** | `VHS_BALANCED_V2` |
| **Author** | Data Engineering Team — PFE 2025/2026 |
| **Date** | 2026-07-01 |
| **Status** | Exploratory — Read-Only — No score recomputed |

---

> **Important:** This notebook is **strictly exploratory and read-only**. It queries `dwh` and `mart` tables but **never writes, modifies, truncates, or drops any table**. The official VHS score (`VHS_BALANCED_V2`) is not changed by running this notebook.

## 1. Introduction

### Context

The STAFFIM vehicle inspection system produces two types of data per vehicle inspection:

**1. Structured fields** — standardized categorical values stored in `dwh.fact_inspection_checkpoint`:
- `valeur_controle`: e.g. `'Bon'`, `'Controle OK'`, `'Defectueux'`, `'Intervention conseillee'`, `'PROPOSITION FAITE'`, `'OUI'`, `'NON'`
- `est_anomalie`, `est_anomalie_critique`: boolean anomaly flags
- `zone_controle`, `checkpoint_code`: standardized identifiers linking to `mart.dim_checkpoint`

**2. Free-text comments** (`commentaire_zone`) — manual notes written by the inspector in the field, in French, often abbreviated, sometimes misspelled.

### How VHS_BALANCED_V2 Currently Works

The Vehicle Health Score is computed **exclusively from structured fields**. The `mart.dim_checkpoint` table assigns criticality tiers and penalty weights to each checkpoint. The scoring engine (`compute_vhs_v2.py`) classifies each checkpoint observation as one of:

| Status | Condition |
|--------|-----------|
| `OK` | `valeur_controle` in `{'Bon', 'Controle OK', 'OUI'}` |
| `WORN` | `est_anomalie=true`, `est_anomalie_critique=false`, `valeur_controle='Intervention conseillee'` |
| `WORN_STRONG` | `est_anomalie_critique=true` AND `valeur_controle='Intervention conseillee'` *(new in V2)* |
| `BROKEN` | `est_anomalie=true`, `valeur_controle` in `{'Defectueux', 'PROPOSITION FAITE', 'NON'}` |
| `IGNORED` | Checkpoint not scored (`is_vhs_scored=false`) |

### Why Comments Are Not Used in the Official Score

| Reason | Detail |
|--------|--------|
| **Inconsistency** | Two inspectors describe the same defect differently |
| **Abbreviations** | `essui` (essuie-glace), `durits` (durites), `retv` (retroviseur), `patin` (plaquette) |
| **Ambiguous scope** | One comment may refer to several checkpoints simultaneously |
| **Already-repaired items** | `"essui glass a changee"` = repair done, not a current defect |
| **Advisory vs defect** | `"vidange conseille"` is a suggestion, not a confirmed failure |
| **Explainability** | Free text reduces auditability and complicates regulatory justification |

### Objective of This Notebook

We explore whether inspector comments can **support or challenge** the interpretation of ambiguous structured values:

- `'PROPOSITION FAITE'` — may indicate a suggestion, not a confirmed defect
- `'Intervention conseillee'` — advisory wording; V2 classifies as WORN_STRONG only when `est_anomalie_critique=true`
- `'Controle OK'` / `'Bon'` accompanied by a comment describing a problem
- CRITIQUE decisions driven by advisory language in comments

**This analysis informs the decision on whether VHS_BALANCED_V3 is needed.**

> **Scope boundary:** No code in `compute_vhs.py` or `compute_vhs_v2.py` is modified. No VHS scores are recomputed. No database table is written.

---
## Setup

In [367]:
import sys
import os
import re
import csv
import unicodedata
import warnings
from pathlib import Path

import pandas as pd

warnings.filterwarnings('ignore')

# Locate project root: notebooks/ is one level below root
NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent
ETL_DWH_DIR  = PROJECT_ROOT / 'etl' / 'dwh'

if str(ETL_DWH_DIR) not in sys.path:
    sys.path.insert(0, str(ETL_DWH_DIR))

pd.set_option('display.max_colwidth', 150)
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)

print(f'Project root : {PROJECT_ROOT}')
print(f'ETL DWH path : {ETL_DWH_DIR}')
print('Libraries loaded.')

Project root : C:\Users\wiem\Downloads\Projet PFE\IRIS_AUTO_FRAUD
ETL DWH path : C:\Users\wiem\Downloads\Projet PFE\IRIS_AUTO_FRAUD\etl\dwh
Libraries loaded.


### Database Connection

The connection reuses the shared `dwh_utils.build_engine()` utility, which reads credentials from the project `.env` file. Passwords are **never exposed in the notebook**.

If `dwh_utils` is unavailable (e.g. running outside the project tree), a safe fallback reads credentials from environment variables.

In [368]:
from sqlalchemy import create_engine, URL, text as sa_text

try:
    import dwh_utils
    engine = dwh_utils.build_engine()
    print('Connected via dwh_utils.build_engine()')
except Exception as exc:
    print(f'dwh_utils unavailable ({exc}) — falling back to environment variables.')
    _url = URL.create(
        drivername='postgresql+psycopg2',
        host=os.environ.get('DB_HOST', 'localhost'),
        port=int(os.environ.get('DB_PORT', 5432)),
        database=os.environ.get('DB_NAME', 'iris_auto_fraud'),
        username=os.environ.get('DB_USER'),
        password=os.environ.get('DB_PASSWORD'),
    )
    engine = create_engine(_url, pool_pre_ping=True)
    print('Connected via environment variables.')

with engine.connect() as conn:
    row = conn.execute(sa_text('SELECT current_database(), current_user, version()')).fetchone()
    print(f'Database : {row[0]}')
    print(f'User     : {row[1]}')
    print(f'Version  : {str(row[2])[:60]}...')

Connected via dwh_utils.build_engine()
Database : iris_auto_fraud
User     : postgres
Version  : PostgreSQL 18.4 on x86_64-windows, compiled by msvc-19.44.35...


### VHS V2 Run Configuration

In [369]:
RUN_ID       = 'VHS_BALANCED_V2_20260630_133318'
PROFILE_NAME = 'VHS_BALANCED_V2'
RULE_VERSION = 'VHS_BALANCED_V2'

OUTPUT_DIR = PROJECT_ROOT / 'data' / 'quality_reports' / 'vhs' / 'staffim_comment_analysis'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Run ID       : {RUN_ID}')
print(f'Profile      : {PROFILE_NAME}')
print(f'Rule Version : {RULE_VERSION}')
print(f'Output dir   : {OUTPUT_DIR}')

Run ID       : VHS_BALANCED_V2_20260630_133318
Profile      : VHS_BALANCED_V2
Rule Version : VHS_BALANCED_V2
Output dir   : C:\Users\wiem\Downloads\Projet PFE\IRIS_AUTO_FRAUD\data\quality_reports\vhs\staffim_comment_analysis


---
## 2. Load Data

We build one flat dataframe `df_staffim` that joins five tables:

| Table | Role |
|-------|------|
| `dwh.fact_inspection_checkpoint` | One row per checkpoint per inspection — `valeur_controle`, `commentaire_zone`, anomaly flags |
| `dwh.fact_inspection_vehicule` | Inspection metadata — vehicle ID, date, kilometrage |
| `mart.dim_checkpoint` | Checkpoint reference — tier, criticality flags, penalty weights |
| `mart.fact_vhs_penalty_detail` | V2 penalty observations — `observed_status_v2`, `penalty_applied_v2` |
| `mart.fact_vhs_score` | V2 score per inspection — `safety_grade_v2`, `decision_v2`, `hard_cap_type_v2` |

The joins on mart tables are **LEFT** so that checkpoints not scored by V2 (e.g. `is_vhs_scored=false`) are still included in the analysis — we need to profile comments on those rows too.

In [370]:
SQL_STAFFIM = '''
SELECT
    fic.inspection_key,
    fiv.immatriculation_norm,
    fiv.date_inspection_sk,
    fiv.kilometrage,
    fic.checkpoint_code,
    dc.checkpoint_libelle,
    dc.zone_controle,
    fic.valeur_controle,
    fic.est_anomalie,
    fic.est_anomalie_critique,
    fic.est_controle_renseigne,
    fic.commentaire_zone,
    dc.tier,
    dc.is_vhs_scored,
    dc.is_vital,
    dc.is_important,
    dc.is_critical_functional,
    dc.is_immobilizing,
    dc.penalty_worn,
    dc.penalty_broken,
    fpd.observed_status  AS observed_status_v2,
    fpd.penalty_applied  AS penalty_applied_v2,
    fvs.safety_grade     AS safety_grade_v2,
    fvs.decision         AS decision_v2,
    fvs.hard_cap_type    AS hard_cap_type_v2
FROM dwh.fact_inspection_checkpoint fic
JOIN dwh.fact_inspection_vehicule fiv
    ON fic.inspection_key = fiv.inspection_key
JOIN mart.dim_checkpoint dc
    ON fic.checkpoint_code = dc.checkpoint_code
LEFT JOIN mart.fact_vhs_penalty_detail fpd
    ON  fic.inspection_key  = fpd.inspection_key
    AND fic.checkpoint_code = fpd.checkpoint_code
    AND fpd.run_id          = :run_id
LEFT JOIN mart.fact_vhs_score fvs
    ON  fic.inspection_key  = fvs.inspection_key
    AND fvs.run_id          = :run_id
'''

with engine.connect() as conn:
    df_staffim = pd.read_sql(
        sa_text(SQL_STAFFIM),
        conn,
        params={'run_id': RUN_ID}
    )

print(f'Loaded {len(df_staffim):,} rows x {df_staffim.shape[1]} columns')
print(f'\nDtypes:')
print(df_staffim.dtypes.to_string())

Loaded 12,298 rows x 25 columns

Dtypes:
inspection_key                str
immatriculation_norm          str
date_inspection_sk          int64
kilometrage               float64
checkpoint_code               str
checkpoint_libelle            str
zone_controle                 str
valeur_controle               str
est_anomalie               object
est_anomalie_critique      object
est_controle_renseigne       bool
commentaire_zone              str
tier                          str
is_vhs_scored                bool
is_vital                     bool
is_important                 bool
is_critical_functional       bool
is_immobilizing              bool
penalty_worn              float64
penalty_broken            float64
observed_status_v2            str
penalty_applied_v2        float64
safety_grade_v2               str
decision_v2                   str
hard_cap_type_v2              str


In [371]:
print('=== Sample rows ===')
df_staffim[[
    'inspection_key', 'immatriculation_norm', 'checkpoint_code',
    'zone_controle', 'valeur_controle', 'est_anomalie',
    'est_anomalie_critique', 'commentaire_zone',
    'tier', 'observed_status_v2', 'decision_v2'
]].head(5)

=== Sample rows ===


,inspection_key,immatriculation_norm,checkpoint_code,zone_controle,valeur_controle,est_anomalie,est_anomalie_critique,commentaire_zone,tier,observed_status_v2,decision_v2
0,1089TU113|20260330|236,1089TU113,autres_prestations_fonctionnement_climatisation,ENTRETIEN,NON,True,False,NaN,T3_COSMETIC,WORN,DEGRADE
1,1089TU113|20260330|236,1089TU113,sous_le_capot_controle_batterie_etat_fixation_et_charge,SOUS_CAPOT,Intervention conseillée,True,False,DEMARREUR A REPARER / REVISION OU REMPLACEMENT DE MOTEUR,T2_CRITICAL,WORN,DEGRADE
2,1089TU113|20260330|236,1089TU113,sous_le_capot_controle_du_niveau_huile_moteur,SOUS_CAPOT,Intervention conseillée,True,False,DEMARREUR A REPARER / REVISION OU REMPLACEMENT DE MOTEUR,T2_CRITICAL,WORN,DEGRADE
3,1089TU113|20260330|236,1089TU113,sous_le_vehicule_controle_etancheite_des_amortisseurs_1,SOUS_VEHICULE,Intervention conseillée,True,True,02 ROTULE DE DIRECTION / SILENT BLOC LAMES RESSORTS COMPLET / BOITIER DIRECTION / AILE AV GAUCHE + PARE A CHOCS COTE DROIT A REPARER,T1_IMPORTANT,WORN_STRONG,DEGRADE
4,1089TU113|20260330|236,1089TU113,sous_le_vehicule_controle_etat_sous_caisse_corrosion_ca,SOUS_VEHICULE,Intervention conseillée,True,True,02 ROTULE DE DIRECTION / SILENT BLOC LAMES RESSORTS COMPLET / BOITIER DIRECTION / AILE AV GAUCHE + PARE A CHOCS COTE DROIT A REPARER,T2_CRITICAL,WORN_STRONG,DEGRADE


### Hard-Cap Trigger Flag

A **hard-cap trigger** is a checkpoint row that directly causes the vehicle-level
score to be capped at CRITIQUE or IMMOBILISE, regardless of other penalty sums.

We try to load `is_hard_cap_trigger` from `mart.fact_vhs_penalty_detail`.
If the column is absent (it may not exist in all schema versions), we derive a proxy:
`observed_status_v2 == "BROKEN"` **and** `tier == "T1"` **and** (`is_vital OR is_important`).

> **Read-only:** no table is written.

In [372]:
try:
    with engine.connect() as _conn:
        _hc = pd.read_sql(
            sa_text('SELECT inspection_key, checkpoint_code, is_hard_cap_trigger FROM mart.fact_vhs_penalty_detail WHERE run_id = :run_id'),
            _conn, params={'run_id': RUN_ID}
        )
    df_staffim = df_staffim.merge(_hc, on=['inspection_key', 'checkpoint_code'], how='left')
    df_staffim['is_hard_cap_trigger'] = df_staffim['is_hard_cap_trigger'].fillna(False).astype(bool)
    print('is_hard_cap_trigger loaded from mart.fact_vhs_penalty_detail')
except Exception as _e:
    print(f'Column not available in DB ({_e}); deriving proxy.')
    _vital = df_staffim.get("is_vital", pd.Series(False, index=df_staffim.index))
    _imp   = df_staffim.get("is_important", pd.Series(False, index=df_staffim.index))
    df_staffim["is_hard_cap_trigger"] = (
        (df_staffim["observed_status_v2"] == "BROKEN")
        & (df_staffim["tier"] == "T1")
        & (_vital | _imp)
    )
    print("is_hard_cap_trigger derived as proxy (BROKEN + T1 + vital/important)")

print("Hard-cap triggers:", int(df_staffim["is_hard_cap_trigger"].sum()))

is_hard_cap_trigger loaded from mart.fact_vhs_penalty_detail
Hard-cap triggers: 372


---
## 3. Basic Profiling

Before analysing comments we establish baseline counts and measure comment coverage across the dataset.

> **Read-only:** all cells below are aggregations only — no table is modified.

In [373]:
# Boolean mask for rows that carry a non-empty, non-blank comment
mask_has_comment = (
    df_staffim['commentaire_zone'].notna()
    & (df_staffim['commentaire_zone'].str.strip() != '')
)

n_total       = len(df_staffim)
n_inspections = df_staffim['inspection_key'].nunique()
n_vehicles    = df_staffim['immatriculation_norm'].nunique()
n_commented   = int(mask_has_comment.sum())
pct_commented = round(100 * n_commented / n_total, 1)

print('=== Global Profile ===')
print(f'  Total checkpoint rows    : {n_total:>10,}')
print(f'  Distinct inspections     : {n_inspections:>10,}')
print(f'  Distinct vehicles        : {n_vehicles:>10,}')
print(f'  Rows with comment        : {n_commented:>10,}  ({pct_commented}%)')
print(f'  Rows without comment     : {n_total - n_commented:>10,}  ({round(100 - pct_commented, 1)}%)')

=== Global Profile ===
  Total checkpoint rows    :     12,298
  Distinct inspections     :        286
  Distinct vehicles        :        280
  Rows with comment        :      3,970  (32.3%)
  Rows without comment     :      8,328  (67.7%)


In [374]:
# Comments by zone_controle (top 20)
df_by_zone = (
    df_staffim[mask_has_comment]
    .groupby('zone_controle', dropna=False)
    .size()
    .reset_index(name='n_with_comment')
    .sort_values('n_with_comment', ascending=False)
    .head(20)
)
print('=== Comments by zone_controle (top 20) ===')
print(df_by_zone.to_string(index=False))

=== Comments by zone_controle (top 20) ===
   zone_controle  n_with_comment
   SOUS_VEHICULE            1612
       INTERIEUR             765
TOUR_DU_VEHICULE             747
      SOUS_CAPOT             534
       ENTRETIEN             312


In [375]:
# Comments by valeur_controle
df_by_valeur = (
    df_staffim[mask_has_comment]
    .groupby('valeur_controle', dropna=False)
    .size()
    .reset_index(name='n_with_comment')
    .sort_values('n_with_comment', ascending=False)
)
print('=== Comments by valeur_controle ===')
print(df_by_valeur.to_string(index=False))

=== Comments by valeur_controle ===
                             valeur_controle  n_with_comment
                                 Contrôle OK            2345
                                         Bon             563
                     Intervention conseillée             436
                                         OUI             207
                                  Défectueux             146
                             Contrôle non OK             124
                                         NON              85
                           Proposition faite              38
                           PROPOSITION FAITE              20
Réparation effectuée suite à l’accord client               4
Réparation effectuée suite à l'accord client               2


In [376]:
# Comments by tier
df_by_tier = (
    df_staffim[mask_has_comment]
    .groupby('tier', dropna=False)
    .size()
    .reset_index(name='n_with_comment')
    .sort_values('n_with_comment', ascending=False)
)
print('=== Comments by tier ===')
print(df_by_tier.to_string(index=False))

=== Comments by tier ===
          tier  n_with_comment
   T2_CRITICAL             863
      T1_VITAL             833
     T2_NORMAL             811
UNKNOWN_REVIEW             810
  T1_IMPORTANT             414
   T3_COSMETIC             239


In [377]:
# Comments by observed_status_v2
df_by_status = (
    df_staffim[mask_has_comment]
    .groupby('observed_status_v2', dropna=False)
    .size()
    .reset_index(name='n_with_comment')
    .sort_values('n_with_comment', ascending=False)
)
print('=== Comments by observed_status_v2 ===')
print(df_by_status.to_string(index=False))

=== Comments by observed_status_v2 ===
observed_status_v2  n_with_comment
               NaN            3249
              WORN             396
       WORN_STRONG             204
            BROKEN             116
          REPAIRED               5


In [378]:
# Comments by decision_v2
df_by_decision = (
    df_staffim[mask_has_comment]
    .groupby('decision_v2', dropna=False)
    .size()
    .reset_index(name='n_with_comment')
    .sort_values('n_with_comment', ascending=False)
)
print('=== Comments by decision_v2 (CRITIQUE / DEGRADE / IMMOBILISE / OK) ===')
print(df_by_decision.to_string(index=False))

=== Comments by decision_v2 (CRITIQUE / DEGRADE / IMMOBILISE / OK) ===
decision_v2  n_with_comment
    DEGRADE            2400
         OK             794
   CRITIQUE             739
 IMMOBILISE              37


In [379]:
# Comment length distribution (non-empty rows only)
lengths = (
    df_staffim
    .loc[mask_has_comment, 'commentaire_zone']
    .str.strip()
    .str.len()
)
print('=== Comment Length Distribution (non-empty rows) ===')
print(lengths.describe().rename({'count': 'n_rows'}).to_string())
print(f'Median : {lengths.median():.0f} characters')
print(f'Max    : {lengths.max()} characters')

# Top 5 longest comments
top5_idx = lengths.nlargest(5).index
print('=== 5 Longest Comments ===')
for idx in top5_idx:
    txt = df_staffim.loc[idx, 'commentaire_zone']
    vc  = df_staffim.loc[idx, 'valeur_controle']
    print(f'  [{vc}] {str(txt)[:120]}')

=== Comment Length Distribution (non-empty rows) ===
n_rows   3970.00
mean       50.36
std        34.50
min         2.00
25%        24.00
50%        39.50
75%        66.75
max       202.00
Median : 40 characters
Max    : 202 characters
=== 5 Longest Comments ===
  [Intervention conseillée] Plaquette avant 2 Amortisseur avant 2 Amortisseur arrière Palier de barra stabilisatrice av 4 Silen bloc bras de suspens
  [Intervention conseillée] Plaquette avant 2 Amortisseur avant 2 Amortisseur arrière Palier de barra stabilisatrice av 4 Silen bloc bras de suspens
  [Intervention conseillée] Plaquette avant 2 Amortisseur avant 2 Amortisseur arrière Palier de barra stabilisatrice av 4 Silen bloc bras de suspens
  [Intervention conseillée] Plaquette avant 2 Amortisseur avant 2 Amortisseur arrière Palier de barra stabilisatrice av 4 Silen bloc bras de suspens
  [Intervention conseillée] Plaquette avant 2 Amortisseur avant 2 Amortisseur arrière Palier de barra stabilisatrice av 4 Silen bloc bras de 

---
## 4. Comment Cleaning

Inspector comments are messy: mixed separators (`/`, `;`, `,`), missing accents, abbreviations, and spelling variants. We apply a **light** normalisation function that prepares comments for keyword matching without destroying business signal.

### Cleaning rules

| Step | Action | Reason |
|------|--------|--------|
| 1 | Lowercase | Uniform matching |
| 2 | Strip accents (NFD) | Catch `essuie` / `essui`, `defectueux` / `defectueux` |
| 3 | Replace `/`, `;`, `,` with ` \| ` | Tokenise compound entries |
| 4 | Collapse repeated spaces | Normalise gaps |
| 5 | Preserve `commentaire_original` | Raw text kept as business evidence |

> Cleaning is **intentionally light** — over-normalisation risks erasing inspector intent that keyword matching relies on.

In [380]:
def _strip_accents(text: str) -> str:
    """Remove diacritics using Unicode NFD decomposition."""
    nfd = unicodedata.normalize('NFD', text)
    return ''.join(c for c in nfd if unicodedata.category(c) != 'Mn')


def clean_staffim_comment(raw) -> str:
    """Light normalisation of commentaire_zone.

    Does NOT over-clean: abbreviations and compound entries are preserved.
    Returns an empty string for null / blank input.
    """
    if raw is None or isinstance(raw, float):
        return ''
    text = str(raw).strip()
    if not text:
        return ''
    text = text.lower()
    text = _strip_accents(text)
    text = re.sub(r'[/;,]+', ' | ', text)   # unify compound separators
    text = re.sub(r'\s+', ' ', text).strip()  # collapse whitespace
    return text


# --- Unit tests with the sample comments from the brief ---
_examples = [
    'essui glass a changee / eclairage ar cote cond /retv gauche / penus ar a change',
    'durits de radiateur a change/niveau huile moteur manque 1l/ batterie a 70%',
    'patin av/ amortisseur av/ fuite huile moteur ( cage soupape et parahuile )',
    'vidange conseille',
    None,
    '',
]
print('=== clean_staffim_comment unit tests ===')
for raw in _examples:
    cleaned = clean_staffim_comment(raw)
    print(f'  IN : {str(raw)[:70]}')
    print(f'  OUT: {cleaned[:70]}')
    print()

=== clean_staffim_comment unit tests ===
  IN : essui glass a changee / eclairage ar cote cond /retv gauche / penus ar
  OUT: essui glass a changee | eclairage ar cote cond | retv gauche | penus a

  IN : durits de radiateur a change/niveau huile moteur manque 1l/ batterie a
  OUT: durits de radiateur a change | niveau huile moteur manque 1l | batteri

  IN : patin av/ amortisseur av/ fuite huile moteur ( cage soupape et parahui
  OUT: patin av | amortisseur av | fuite huile moteur ( cage soupape et parah

  IN : vidange conseille
  OUT: vidange conseille

  IN : None
  OUT: 

  IN : 
  OUT: 



In [381]:
# Preserve original; create cleaned variant and derived columns
df_staffim['commentaire_original'] = df_staffim['commentaire_zone']
df_staffim['commentaire_clean']    = df_staffim['commentaire_zone'].apply(clean_staffim_comment)
df_staffim['has_comment']          = df_staffim['commentaire_clean'].str.strip().astype(bool)
df_staffim['comment_length']       = df_staffim['commentaire_clean'].str.len()

# Token list: words from cleaned comment, excluding the "|" separators
df_staffim['comment_tokens'] = df_staffim['commentaire_clean'].apply(
    lambda x: [t for t in x.split() if t != '|'] if x else []
)

n_clean = int(df_staffim['has_comment'].sum())
print(f'Rows with non-empty cleaned comment : {n_clean:,}')
print(f'Rows without comment                : {len(df_staffim) - n_clean:,}')

Rows with non-empty cleaned comment : 3,970
Rows without comment                : 8,328


In [382]:
# Side-by-side view: original vs cleaned for non-empty rows
preview_cols = [
    'inspection_key', 'checkpoint_code', 'valeur_controle',
    'commentaire_original', 'commentaire_clean', 'comment_length'
]
df_staffim[df_staffim['has_comment']][preview_cols].head(10)

,inspection_key,checkpoint_code,valeur_controle,commentaire_original,commentaire_clean,comment_length
1,1089TU113|20260330|236,sous_le_capot_controle_batterie_etat_fixation_et_charge,Intervention conseillée,DEMARREUR A REPARER / REVISION OU REMPLACEMENT DE MOTEUR,demarreur a reparer | revision ou remplacement de moteur,56
2,1089TU113|20260330|236,sous_le_capot_controle_du_niveau_huile_moteur,Intervention conseillée,DEMARREUR A REPARER / REVISION OU REMPLACEMENT DE MOTEUR,demarreur a reparer | revision ou remplacement de moteur,56
3,1089TU113|20260330|236,sous_le_vehicule_controle_etancheite_des_amortisseurs_1,Intervention conseillée,02 ROTULE DE DIRECTION / SILENT BLOC LAMES RESSORTS COMPLET / BOITIER DIRECTION / AILE AV GAUCHE + PARE A CHOCS COTE DROIT A REPARER,02 rotule de direction | silent bloc lames ressorts complet | boitier direction | aile av gauche + pare a chocs cote droit a reparer,132
4,1089TU113|20260330|236,sous_le_vehicule_controle_etat_sous_caisse_corrosion_ca,Intervention conseillée,02 ROTULE DE DIRECTION / SILENT BLOC LAMES RESSORTS COMPLET / BOITIER DIRECTION / AILE AV GAUCHE + PARE A CHOCS COTE DROIT A REPARER,02 rotule de direction | silent bloc lames ressorts complet | boitier direction | aile av gauche + pare a chocs cote droit a reparer,132
5,1089TU113|20260330|236,sous_le_vehicule_controle_gaine_transmissions_rotules_c,Intervention conseillée,02 ROTULE DE DIRECTION / SILENT BLOC LAMES RESSORTS COMPLET / BOITIER DIRECTION / AILE AV GAUCHE + PARE A CHOCS COTE DROIT A REPARER,02 rotule de direction | silent bloc lames ressorts complet | boitier direction | aile av gauche + pare a chocs cote droit a reparer,132
6,1125TU116|20260313|217,sous_le_capot_controle_du_niveau_du_liquide_de_refroidi,Intervention conseillée,huile moteur niveau min liquide refroidissement niveau min courroie accessoire a remplacer,huile moteur niveau min liquide refroidissement niveau min courroie accessoire a remplacer,90
7,1125TU116|20260313|217,sous_le_capot_controle_du_niveau_huile_moteur,Intervention conseillée,huile moteur niveau min liquide refroidissement niveau min courroie accessoire a remplacer,huile moteur niveau min liquide refroidissement niveau min courroie accessoire a remplacer,90
8,1125TU116|20260313|217,sous_le_capot_controle_etat_des_courroies_d_accessoires,Contrôle non OK,huile moteur niveau min liquide refroidissement niveau min courroie accessoire a remplacer,huile moteur niveau min liquide refroidissement niveau min courroie accessoire a remplacer,90
9,1125TU116|20260313|217,sous_le_vehicule_controle_des_plaquettes_de_freins_av,Intervention conseillée,plaquette ava remplacer fuite huile joint cache soupape,plaquette ava remplacer fuite huile joint cache soupape,55
10,1125TU116|20260313|217,sous_le_vehicule_controle_etancheite_tous_fluides,Intervention conseillée,plaquette ava remplacer fuite huile joint cache soupape,plaquette ava remplacer fuite huile joint cache soupape,55


In [383]:
from collections import Counter

lengths     = df_staffim.loc[df_staffim['has_comment'], 'comment_length']
token_counts = df_staffim.loc[df_staffim['has_comment'], 'comment_tokens'].str.len()

'=== Comment Length (chars) — non-empty rows ==='
print('=== Comment Length (chars) — non-empty rows ===')
print(lengths.describe().to_string())

print('\n=== Token Count per Comment — non-empty rows ===')
print(token_counts.describe().to_string())

# Most frequent single tokens across all comments
all_tokens = [
    tok
    for tokens in df_staffim.loc[df_staffim['has_comment'], 'comment_tokens']
    for tok in tokens
]
top_tokens = Counter(all_tokens).most_common(30)
print('\n=== Top 30 Tokens Across All Comments ===')
for tok, cnt in top_tokens:
    print(f'  {tok:<25} : {cnt:>5,}')

=== Comment Length (chars) — non-empty rows ===
count   3970.00
mean      50.56
std       34.66
min        2.00
25%       24.00
50%       40.00
75%       66.75
max      204.00

=== Token Count per Comment — non-empty rows ===
count   3970.00
mean       8.13
std        5.48
min        1.00
25%        4.00
50%        6.00
75%       11.00
max       31.00

=== Top 30 Tokens Across All Comments ===
  a                         : 2,204
  remplacer                 : 1,608
  av                        : 1,553
  de                        :   974
  ar                        :   912
  huile                     :   634
  glace                     :   627
  essuie                    :   609
  et                        :   558
  pas                       :   556
  moteur                    :   554
  plaquette                 :   542
  fuite                     :   503
  direction                 :   474
  ne                        :   468
  2                         :   450
  rotule                   

---
## 5. Keyword Dictionary Exploration

We define a domain-oriented keyword dictionary organised by vehicle component group. Each group maps to a boolean column on `df_staffim`.

> **Scope:** This dictionary is an **audit tool only** — it is never used as a direct VHS scoring input. Keyword matches flag rows for human review; they do not change any penalty or decision.

### Dictionary groups

| Column | Domain | Example keywords |
|--------|--------|-----------------|
| `kw_brake` | Braking system | frein, plaquette, disque, liquide frein |
| `kw_tyre` | Tyres / wheels | pneu, roue, pression |
| `kw_suspension_direction` | Suspension & steering | amortisseur, rotule, transmission, direction |
| `kw_engine_fluids` | Engine & fluids | huile, moteur, fuite, batterie, courroie, durite |
| `kw_lighting_visibility` | Lighting & visibility | feu, eclairage, stop, phare, essuie, retro |
| `kw_action_repair` | Action / severity words | a changer, manque, fuite, casse, defectueux, proposition |
| `kw_leak` | Leak specifically | fuite |
| `kw_missing_level` | Missing fluid level | manque, niveau.*manque |
| `kw_change_required` | Part replacement needed | a changer, a change, remplace |
| `kw_advisory` | Advisory / suggestion language | conseille, vidange, proposition, intervention |

In [384]:
# All patterns matched against commentaire_clean (lowercased, accent-stripped)
# re.search() is used so partial word matches are allowed (e.g. "frein" matches "freinage")

KEYWORDS = {
    'kw_brake': [
        r'frein', r'plaquette', r'plaquettes', r'disque', r'disques',
        r'liquide frein', r'liquide de frein',
    ],
    'kw_tyre': [
        r'pneu', r'pneus', r'roue', r'pression',
    ],
    'kw_suspension_direction': [
        r'amortisseur', r'amortisseurs', r'rotule', r'rotules',
        r'transmission', r'direction',
    ],
    'kw_engine_fluids': [
        r'huile', r'moteur', r'fuite', r'etancheite',
        r'liquide', r'refroidissement', r'batterie', r'courroie',
        r'durite', r'durits',
    ],
    'kw_lighting_visibility': [
        r'feu', r'feux', r'eclairage', r'stop', r'clignotant',
        r'phare', r'vitre', r'parebrise', r'essuie', r'essui',
        r'retro', r'retroviseur',
    ],
    'kw_action_repair': [
        r'a changer', r'a change', r'manque', r'fuite', r'casse',
        r'defectueux', r'non ok', r'conseille', r'proposition', r'vidange',
    ],
    'kw_leak': [
        r'fuite',
    ],
    'kw_missing_level': [
        r'manque', r'niveau.{0,10}manque', r'manque.{0,10}niveau', r'manque.{0,5}l\b',
    ],
    'kw_change_required': [
        r'a changer', r'a change', r'remplace', r'remplacement',
    ],
    'kw_advisory': [
        r'conseille', r'vidange conseille', r'proposition', r'intervention',
    ],
}


def has_keyword(text: str, patterns: list) -> bool:
    """Return True if any pattern matches within the cleaned comment."""
    if not text:
        return False
    for pat in patterns:
        if re.search(pat, text):
            return True
    return False


# Apply keyword boolean columns
for col, patterns in KEYWORDS.items():
    df_staffim[col] = df_staffim['commentaire_clean'].apply(
        lambda x, p=patterns: has_keyword(x, p)
    )

print('Keyword columns created.')

Keyword columns created.


In [385]:
n_total = len(df_staffim)
print('=== Keyword coverage — all rows ===')
print(f'  {"Column":<35}  {"Count":>7}  {"% all rows":>10}')
print('  ' + '-' * 57)
for col in KEYWORDS:
    n   = int(df_staffim[col].sum())
    pct = 100 * n / n_total
    print(f'  {col:<35}  {n:>7,}  {pct:>9.1f}%')

=== Keyword coverage — all rows ===
  Column                                 Count  % all rows
  ---------------------------------------------------------
  kw_brake                                 749        6.1%
  kw_tyre                                  431        3.5%
  kw_suspension_direction                  825        6.7%
  kw_engine_fluids                       1,220        9.9%
  kw_lighting_visibility                 1,149        9.3%
  kw_action_repair                       1,132        9.2%
  kw_leak                                  464        3.8%
  kw_missing_level                         175        1.4%
  kw_change_required                     1,615       13.1%
  kw_advisory                               42        0.3%


In [386]:
df_commented = df_staffim[df_staffim['has_comment']].copy()
n_commented  = len(df_commented)

print(f'=== Keyword coverage — {n_commented:,} rows WITH a comment ===')
print(f'  {"Column":<35}  {"Count":>7}  {"% commented":>11}')
print('  ' + '-' * 59)
for col in KEYWORDS:
    n   = int(df_commented[col].sum())
    pct = 100 * n / n_commented if n_commented else 0
    print(f'  {col:<35}  {n:>7,}  {pct:>10.1f}%')

=== Keyword coverage — 3,970 rows WITH a comment ===
  Column                                 Count  % commented
  -----------------------------------------------------------
  kw_brake                                 749        18.9%
  kw_tyre                                  431        10.9%
  kw_suspension_direction                  825        20.8%
  kw_engine_fluids                       1,220        30.7%
  kw_lighting_visibility                 1,149        28.9%
  kw_action_repair                       1,132        28.5%
  kw_leak                                  464        11.7%
  kw_missing_level                         175         4.4%
  kw_change_required                     1,615        40.7%
  kw_advisory                               42         1.1%


In [387]:
# Co-occurrence: how often do keyword groups appear together (among commented rows)?
kw_cols = list(KEYWORDS.keys())
cooc = df_commented[kw_cols].astype(int)

# Build upper-triangle co-occurrence counts
cooc_matrix = cooc.T.dot(cooc)

print('=== Keyword Co-occurrence Matrix (commented rows) ===')
print('(diagonal = rows matching that keyword group)')
print(cooc_matrix.to_string())

=== Keyword Co-occurrence Matrix (commented rows) ===
(diagonal = rows matching that keyword group)
                         kw_brake  kw_tyre  kw_suspension_direction  kw_engine_fluids  kw_lighting_visibility  kw_action_repair  kw_leak  kw_missing_level  kw_change_required  kw_advisory
kw_brake                      749       22                      312               276                       0               258      207                12                 456            0
kw_tyre                        22      431                       39                22                     153                84       13                22                 148            0
kw_suspension_direction       312       39                      825               344                       0               364      299                13                 442            0
kw_engine_fluids              276       22                      344              1220                     204               684      464            

In [388]:
# Which valeur_controle values most often carry each keyword?
# Focus on safety-critical keyword groups.
focus_kw = ['kw_brake', 'kw_suspension_direction', 'kw_leak',
            'kw_change_required', 'kw_advisory']

print('=== Safety keyword frequency by valeur_controle (commented rows) ===')
for col in focus_kw:
    print(f'\n  -- {col} --')
    sub = (
        df_commented[df_commented[col]]
        .groupby('valeur_controle', dropna=False)
        .size()
        .reset_index(name='n')
        .sort_values('n', ascending=False)
    )
    print(sub.to_string(index=False))

=== Safety keyword frequency by valeur_controle (commented rows) ===

  -- kw_brake --
                             valeur_controle   n
                                 Contrôle OK 569
                     Intervention conseillée 116
                             Contrôle non OK  42
                                         Bon  15
                                  Défectueux   3
Réparation effectuée suite à l’accord client   3
Réparation effectuée suite à l'accord client   1

  -- kw_suspension_direction --
                             valeur_controle   n
                                 Contrôle OK 649
                     Intervention conseillée 137
                             Contrôle non OK  36
Réparation effectuée suite à l’accord client   3

  -- kw_leak --
                             valeur_controle   n
                                 Contrôle OK 343
                     Intervention conseillée  86
                             Contrôle non OK  32
Réparation effectuée suite à l

In [389]:
# Show 3 real comment examples for each keyword group
print('=== Sample comments per keyword group ===')
for col in KEYWORDS:
    hits = df_commented[df_commented[col]]
    if hits.empty:
        print(f'\n[{col}] — no matches')
        continue
    print(f'\n[{col}] — {len(hits):,} matches — first 3 examples:')
    for _, row in hits[['valeur_controle', 'commentaire_original']].head(3).iterrows():
        vc  = str(row['valeur_controle'])[:25]
        txt = str(row['commentaire_original'])[:100]
        print(f'  [{vc}]  {txt}')

=== Sample comments per keyword group ===

[kw_brake] — 749 matches — first 3 examples:
  [Intervention conseillée]  plaquette ava remplacer fuite huile joint cache soupape
  [Intervention conseillée]  plaquette ava remplacer fuite huile joint cache soupape
  [Défectueux]  Plaquette police av

[kw_tyre] — 431 matches — first 3 examples:
  [Proposition faite]  4 PNEU USEE
  [Proposition faite]  4 PNEU USEE
  [Proposition faite]  4 PNEU USEE

[kw_suspension_direction] — 825 matches — first 3 examples:
  [Intervention conseillée]  02 ROTULE DE DIRECTION / SILENT BLOC LAMES RESSORTS COMPLET / BOITIER DIRECTION / AILE AV GAUCHE + P
  [Intervention conseillée]  02 ROTULE DE DIRECTION / SILENT BLOC LAMES RESSORTS COMPLET / BOITIER DIRECTION / AILE AV GAUCHE + P
  [Intervention conseillée]  02 ROTULE DE DIRECTION / SILENT BLOC LAMES RESSORTS COMPLET / BOITIER DIRECTION / AILE AV GAUCHE + P

[kw_engine_fluids] — 1,220 matches — first 3 examples:
  [Intervention conseillée]  DEMARREUR A REPARER 

In [390]:
# Build detected_keywords: comma-separated labels of matched keyword groups
_KW_LABELS = {
    'kw_brake':               'brake',
    'kw_tyre':                'tyre',
    'kw_suspension_direction':'suspension',
    'kw_engine_fluids':       'engine_fluids',
    'kw_lighting_visibility': 'lighting',
    'kw_action_repair':       'action_repair',
    'kw_leak':                'leak',
    'kw_missing_level':       'missing_level',
    'kw_change_required':     'change_required',
    'kw_advisory':            'advisory',
}

def get_detected_keywords(row) -> str:
    hits = [label for col, label in _KW_LABELS.items() if row.get(col, False)]
    return ', '.join(hits) if hits else ''

df_staffim['detected_keywords'] = df_staffim.apply(get_detected_keywords, axis=1)

# Quick coverage check
n_any_kw = (df_staffim['detected_keywords'] != '').sum()
print(f'Rows with at least one keyword hit: {n_any_kw:,} ({100*n_any_kw/len(df_staffim):.1f}%)')
print(df_staffim[['commentaire_clean', 'detected_keywords']].head(5).to_string())

Rows with at least one keyword hit: 3,544 (28.8%)
                                                                                                                      commentaire_clean               detected_keywords
0                                                                                                                                                                      
1                                                                              demarreur a reparer | revision ou remplacement de moteur  engine_fluids, change_required
2                                                                              demarreur a reparer | revision ou remplacement de moteur  engine_fluids, change_required
3  02 rotule de direction | silent bloc lames ressorts complet | boitier direction | aile av gauche + pare a chocs cote droit a reparer                      suspension
4  02 rotule de direction | silent bloc lames ressorts complet | boitier direction | aile av gauche + pare a c

---
## 6. Comment vs Structured Value Consistency

We build five audit dataframes — one per scenario — that surface potential inconsistencies between `valeur_controle` (structured) and `commentaire_zone` (free-text).

| Audit | Question |
|-------|---------|
| **A** | Rows where structured value says OK but comment mentions a defect |
| **B** | All `PROPOSITION FAITE` rows — advisory vs confirmed defect |
| **C** | All `Intervention conseillee` rows — WORN_STRONG coverage and comment context |
| **D** | CRITIQUE decisions where comments use only advisory language |
| **E** | CRITIQUE / IMMOBILISE decisions confirmed by strong defect evidence in comments |

> **Read-only:** these dataframes are views of `df_staffim`. Nothing is written to the database.

In [391]:
# Structured "OK" values — same set used by the scoring engine
OK_VALUES = {'Bon', 'Controle OK', 'Contrôle OK', 'OUI'}

# Any defect-related keyword in the comment
_kw_any_defect = (
    df_staffim['kw_brake']
    | df_staffim['kw_suspension_direction']
    | df_staffim['kw_engine_fluids']
    | df_staffim['kw_leak']
    | df_staffim['kw_change_required']
    | df_staffim['kw_missing_level']
)

# Strong defect: part failure, leak, missing level, replacement needed
_kw_strong_defect = (
    df_staffim['kw_brake']
    | df_staffim['kw_suspension_direction']
    | df_staffim['kw_leak']
    | df_staffim['kw_change_required']
    | df_staffim['kw_missing_level']
)

print('Shared masks computed.')

Shared masks computed.


### Audit A — OK Structured Value with Defect Keyword in Comment

These rows present a **signal inconsistency**: the structured field says the checkpoint passed, but the inspector wrote a comment that mentions a defect or repair action.

Possible explanations:
- The inspector noted a repair *already performed* (e.g. `"essui glass a changee"`) — not a current defect.
- The comment refers to a different checkpoint than the one being observed.
- The structured value was mis-coded.

This audit helps quantify how frequent such mismatches are.

In [392]:
AUDIT_A_COLS = [
    'inspection_key', 'immatriculation_norm', 'checkpoint_code',
    'checkpoint_libelle', 'zone_controle', 'tier',
    'valeur_controle', 'est_anomalie', 'est_anomalie_critique',
    'commentaire_original', 'commentaire_clean',
    'observed_status_v2', 'penalty_applied_v2',
    'safety_grade_v2', 'decision_v2',
    'kw_brake', 'kw_engine_fluids', 'kw_leak',
    'kw_change_required', 'kw_missing_level',
]

mask_audit_a = (
    df_staffim['valeur_controle'].isin(OK_VALUES)
    & df_staffim['has_comment']
    & _kw_any_defect
)

comments_with_ok_value = (
    df_staffim[mask_audit_a][AUDIT_A_COLS]
    .reset_index(drop=True)
)

print(f'Audit A — OK value with defect keyword in comment: {len(comments_with_ok_value):,} rows')
print(f"  kw_brake         : {comments_with_ok_value['kw_brake'].sum()}")
print(f"  kw_engine_fluids : {comments_with_ok_value['kw_engine_fluids'].sum()}")
print(f"  kw_leak          : {comments_with_ok_value['kw_leak'].sum()}")
print(f"  kw_change_required: {comments_with_ok_value['kw_change_required'].sum()}")
print(f"  kw_missing_level  : {comments_with_ok_value['kw_missing_level'].sum()}")

comments_with_ok_value.head(8)

Audit A — OK value with defect keyword in comment: 2,119 rows
  kw_brake         : 584
  kw_engine_fluids : 910
  kw_leak          : 343
  kw_change_required: 1255
  kw_missing_level  : 130


,inspection_key,immatriculation_norm,checkpoint_code,checkpoint_libelle,zone_controle,tier,valeur_controle,est_anomalie,est_anomalie_critique,commentaire_original,commentaire_clean,observed_status_v2,penalty_applied_v2,safety_grade_v2,decision_v2,kw_brake,kw_engine_fluids,kw_leak,kw_change_required,kw_missing_level
0,9432TU156|20250901|22,9432TU156,sous_le_capot_controle_batterie_etat_fixation_et_charge,Controle batterie etat fixation et charge,SOUS_CAPOT,T2_CRITICAL,Contrôle OK,False,None,Bouchon vase d'eau à remplacer,bouchon vase d'eau a remplacer,NaN,NaN,B,DEGRADE,False,False,False,True,False
1,3475TU146|20250918|56,3475TU146,dans_le_vehicule_controle_fonctionnement_des_feux_ecl_1,Controle fonctionnement des feux ecl 1,INTERIEUR,UNKNOWN_REVIEW,Contrôle OK,False,None,Manque liquide essuie glace 2 veilleuse av ne fonctionne pas,manque liquide essuie glace 2 veilleuse av ne fonctionne pas,NaN,NaN,C,DEGRADE,False,True,False,False,True
2,8613TU256|20260331|237,8613TU256,sous_le_vehicule_controle_ligne_d_echappement_et_fixati,Controle ligne d echappement et fixati,SOUS_VEHICULE,T2_CRITICAL,Contrôle OK,False,None,02 AMORTISSEURS AR + 02 ANTICHOCS ARRIERE A REMPLACER,02 amortisseurs ar + 02 antichocs arriere a remplacer,NaN,NaN,C,DEGRADE,False,False,False,True,False
3,8041TU243|20260514|283,8041TU243,tour_du_vehicule_vitres_et_pare_brise,Vitres et pare brise,TOUR_DU_VEHICULE,T2_NORMAL,Bon,False,None,PNEUS AVANT A REMPLACER,pneus avant a remplacer,NaN,NaN,C,DEGRADE,False,False,False,True,False
4,5660TU191|20260327|230,5660TU191,sous_le_vehicule_controle_etancheite_tous_fluides,Controle etancheite tous fluides,SOUS_VEHICULE,T2_CRITICAL,Contrôle OK,False,None,rotule pivot droite a remplacer,rotule pivot droite a remplacer,NaN,NaN,C,DEGRADE,False,False,False,True,False
5,7918TU117|20251225|147,7918TU117,sous_le_vehicule_controle_des_plaquettes_de_freins_av,Controle des plaquettes de freins av,SOUS_VEHICULE,T1_VITAL,Contrôle OK,False,None,Jeu et bruit au niveau etriers de frein av,jeu et bruit au niveau etriers de frein av,NaN,NaN,C,DEGRADE,True,False,False,False,False
6,1575TU56|20251104|110,1575TU56,sous_le_vehicule_controle_ligne_d_echappement_et_fixati,Controle ligne d echappement et fixati,SOUS_VEHICULE,T2_CRITICAL,Contrôle OK,False,None,Fuite huile boîte ( parahuile cardon ) Fuite huile Carter Renifleur a remplacer Support moteur amortisseur arriere a remplacer,fuite huile boite ( parahuile cardon ) fuite huile carter renifleur a remplacer support moteur amortisseur arriere a remplacer,NaN,NaN,B,DEGRADE,False,True,True,True,False
7,7456TU157|20250811|6,7456TU157,dans_le_vehicule_controle_fonctionnement_des_feux_de_si,Controle fonctionnement des feux de si,INTERIEUR,T2_NORMAL,Contrôle OK,False,None,manque liquide essuie glace,manque liquide essuie glace,NaN,NaN,D,CRITIQUE,False,True,False,False,True


### Audit B — `PROPOSITION FAITE`: Advisory vs Confirmed Defect

`PROPOSITION FAITE` is a key ambiguous value. The scoring engine currently treats it as `BROKEN`, but the wording implies the inspector *proposed* an action rather than confirming a failure.

We split the rows into two sub-groups based on comment content:
- **Advisory only** — comment contains advisory language but no strong defect word
- **Strong defect** — comment mentions a confirmed defect (leak, part to replace, etc.)

This split will directly inform the V3 decision.

In [393]:
mask_prop = df_staffim['valeur_controle'].str.contains(
    'PROPOSITION FAITE|Proposition faite|proposition faite',
    case=False, na=False, regex=True
)

AUDIT_B_COLS = [
    'inspection_key', 'immatriculation_norm', 'checkpoint_code',
    'checkpoint_libelle', 'zone_controle', 'tier',
    'valeur_controle', 'est_anomalie', 'est_anomalie_critique',
    'commentaire_original', 'commentaire_clean', 'has_comment',
    'observed_status_v2', 'penalty_applied_v2',
    'safety_grade_v2', 'decision_v2',
    'kw_advisory', 'kw_brake', 'kw_engine_fluids',
    'kw_suspension_direction', 'kw_leak', 'kw_change_required', 'kw_missing_level',
]

proposition_faite_comment_context = (
    df_staffim[mask_prop][AUDIT_B_COLS]
    .reset_index(drop=True)
)

# Sub-classify by comment evidence
_prop_has_strong = (
    proposition_faite_comment_context['kw_brake']
    | proposition_faite_comment_context['kw_leak']
    | proposition_faite_comment_context['kw_change_required']
    | proposition_faite_comment_context['kw_missing_level']
    | proposition_faite_comment_context['kw_engine_fluids']
)

n_prop       = len(proposition_faite_comment_context)
n_prop_cmt   = int(proposition_faite_comment_context['has_comment'].sum())
n_prop_adv   = int(
    (proposition_faite_comment_context['has_comment']
     & proposition_faite_comment_context['kw_advisory']
     & ~_prop_has_strong).sum()
)
n_prop_strong = int(
    (proposition_faite_comment_context['has_comment'] & _prop_has_strong).sum()
)

print(f'Audit B — PROPOSITION FAITE rows : {n_prop:,}')
print(f'  With any comment               : {n_prop_cmt:,}')
print(f'  Advisory comment only          : {n_prop_adv:,}')
print(f'  Strong defect in comment       : {n_prop_strong:,}')
print(f'  No comment                     : {n_prop - n_prop_cmt:,}')

print('\n--- Rows with advisory comment (no strong defect) ---')
(
    proposition_faite_comment_context[
        proposition_faite_comment_context['has_comment']
        & proposition_faite_comment_context['kw_advisory']
        & ~_prop_has_strong
    ][[
        'checkpoint_libelle', 'tier', 'valeur_controle',
        'commentaire_original', 'observed_status_v2', 'decision_v2',
    ]].head(5)
)

Audit B — PROPOSITION FAITE rows : 105
  With any comment               : 58
  Advisory comment only          : 2
  Strong defect in comment       : 27
  No comment                     : 47

--- Rows with advisory comment (no strong defect) ---


,checkpoint_libelle,tier,valeur_controle,commentaire_original,observed_status_v2,decision_v2
51,Courroie de disribution,T2_CRITICAL,PROPOSITION FAITE,vidange conseillé,WORN,DEGRADE
104,Operation d entretien,UNKNOWN_REVIEW,PROPOSITION FAITE,VIDANGE CONSEILLE,NaN,OK


### Audit C — `Intervention conseillee`: WORN_STRONG Coverage and Comment Context

In V2, `Intervention conseillee` with `est_anomalie_critique=true` is classified as `WORN_STRONG` — the new status introduced to handle critically-flagged advisory observations more severely than plain `WORN`.

We examine comments to assess whether they support or challenge this escalation:
- Does the comment contain strong defect evidence (supporting WORN_STRONG)?
- Or does the comment use only advisory language (suggesting plain WORN may suffice)?

In [394]:
mask_interv = df_staffim['valeur_controle'].isin(
    ['Intervention conseillee', 'Intervention conseillée']
)

AUDIT_C_COLS = [
    'inspection_key', 'immatriculation_norm', 'checkpoint_code',
    'checkpoint_libelle', 'zone_controle', 'tier',
    'valeur_controle', 'est_anomalie', 'est_anomalie_critique',
    'commentaire_original', 'commentaire_clean', 'has_comment',
    'observed_status_v2', 'penalty_applied_v2',
    'safety_grade_v2', 'decision_v2',
    'kw_brake', 'kw_suspension_direction', 'kw_engine_fluids',
    'kw_leak', 'kw_change_required', 'kw_advisory',
]

intervention_conseillee_comment_context = (
    df_staffim[mask_interv][AUDIT_C_COLS]
    .reset_index(drop=True)
)

intervention_conseillee_comment_context['is_worn_strong'] = (
    intervention_conseillee_comment_context['observed_status_v2'] == 'WORN_STRONG'
)

_interv_strong = (
    intervention_conseillee_comment_context['kw_brake']
    | intervention_conseillee_comment_context['kw_suspension_direction']
    | intervention_conseillee_comment_context['kw_leak']
    | intervention_conseillee_comment_context['kw_change_required']
)

n_interv          = len(intervention_conseillee_comment_context)
n_worn_strong     = int(intervention_conseillee_comment_context['is_worn_strong'].sum())
n_interv_cmt      = int(intervention_conseillee_comment_context['has_comment'].sum())
n_interv_strong   = int((intervention_conseillee_comment_context['has_comment'] & _interv_strong).sum())
n_interv_advisory = int(
    (intervention_conseillee_comment_context['has_comment']
     & intervention_conseillee_comment_context['kw_advisory']
     & ~_interv_strong).sum()
)

print(f'Audit C — Intervention conseillee rows : {n_interv:,}')
print(f'  Classified WORN_STRONG (V2)          : {n_worn_strong:,}')
print(f'  Classified WORN (V2)                 : {n_interv - n_worn_strong:,}')
print(f'  With comment                         : {n_interv_cmt:,}')
print(f'  Comment = strong defect              : {n_interv_strong:,}')
print(f'  Comment = advisory only              : {n_interv_advisory:,}')

intervention_conseillee_comment_context.head(6)

Audit C — Intervention conseillee rows : 575
  Classified WORN_STRONG (V2)          : 269
  Classified WORN (V2)                 : 306
  With comment                         : 436
  Comment = strong defect              : 279
  Comment = advisory only              : 4


,inspection_key,immatriculation_norm,checkpoint_code,checkpoint_libelle,zone_controle,tier,valeur_controle,est_anomalie,est_anomalie_critique,commentaire_original,commentaire_clean,has_comment,observed_status_v2,penalty_applied_v2,safety_grade_v2,decision_v2,kw_brake,kw_suspension_direction,kw_engine_fluids,kw_leak,kw_change_required,kw_advisory,is_worn_strong
0,1089TU113|20260330|236,1089TU113,sous_le_capot_controle_batterie_etat_fixation_et_charge,Controle batterie etat fixation et charge,SOUS_CAPOT,T2_CRITICAL,Intervention conseillée,True,False,DEMARREUR A REPARER / REVISION OU REMPLACEMENT DE MOTEUR,demarreur a reparer | revision ou remplacement de moteur,True,WORN,5.00,C,DEGRADE,False,False,True,False,True,False,False
1,1089TU113|20260330|236,1089TU113,sous_le_capot_controle_du_niveau_huile_moteur,Controle du niveau huile moteur,SOUS_CAPOT,T2_CRITICAL,Intervention conseillée,True,False,DEMARREUR A REPARER / REVISION OU REMPLACEMENT DE MOTEUR,demarreur a reparer | revision ou remplacement de moteur,True,WORN,5.00,C,DEGRADE,False,False,True,False,True,False,False
2,1089TU113|20260330|236,1089TU113,sous_le_vehicule_controle_etancheite_des_amortisseurs_1,Controle etancheite des amortisseurs 1,SOUS_VEHICULE,T1_IMPORTANT,Intervention conseillée,True,True,02 ROTULE DE DIRECTION / SILENT BLOC LAMES RESSORTS COMPLET / BOITIER DIRECTION / AILE AV GAUCHE + PARE A CHOCS COTE DROIT A REPARER,02 rotule de direction | silent bloc lames ressorts complet | boitier direction | aile av gauche + pare a chocs cote droit a reparer,True,WORN_STRONG,10.50,C,DEGRADE,False,True,False,False,False,False,True
3,1089TU113|20260330|236,1089TU113,sous_le_vehicule_controle_etat_sous_caisse_corrosion_ca,Controle etat sous caisse corrosion ca,SOUS_VEHICULE,T2_CRITICAL,Intervention conseillée,True,True,02 ROTULE DE DIRECTION / SILENT BLOC LAMES RESSORTS COMPLET / BOITIER DIRECTION / AILE AV GAUCHE + PARE A CHOCS COTE DROIT A REPARER,02 rotule de direction | silent bloc lames ressorts complet | boitier direction | aile av gauche + pare a chocs cote droit a reparer,True,WORN_STRONG,8.50,C,DEGRADE,False,True,False,False,False,False,True
4,1089TU113|20260330|236,1089TU113,sous_le_vehicule_controle_gaine_transmissions_rotules_c,Controle gaine transmissions rotules c,SOUS_VEHICULE,T1_VITAL,Intervention conseillée,True,True,02 ROTULE DE DIRECTION / SILENT BLOC LAMES RESSORTS COMPLET / BOITIER DIRECTION / AILE AV GAUCHE + PARE A CHOCS COTE DROIT A REPARER,02 rotule de direction | silent bloc lames ressorts complet | boitier direction | aile av gauche + pare a chocs cote droit a reparer,True,WORN_STRONG,17.50,C,DEGRADE,False,True,False,False,False,False,True
5,1125TU116|20260313|217,1125TU116,sous_le_capot_controle_du_niveau_du_liquide_de_refroidi,Controle du niveau du liquide de refroidi,SOUS_CAPOT,T2_CRITICAL,Intervention conseillée,True,False,huile moteur niveau min liquide refroidissement niveau min courroie accessoire a remplacer,huile moteur niveau min liquide refroidissement niveau min courroie accessoire a remplacer,True,WORN,5.00,C,DEGRADE,False,False,True,False,True,False,False


### Audit D — CRITIQUE Decision with Advisory-Only Comments

Some inspections reach a `CRITIQUE` safety decision even when comments use advisory language (`conseille`, `proposition`, `vidange conseille`). This audit surfaces cases where the comment tone is soft but the structured value triggered a severe classification.

This does **not** mean the CRITIQUE decision is wrong — the structured value may correctly reflect the severity. But it flags cases worth human review.

In [395]:
mask_critique_advisory = (
    (df_staffim['decision_v2'] == 'CRITIQUE')
    & df_staffim['has_comment']
    & df_staffim['kw_advisory']
    & ~_kw_strong_defect
)

AUDIT_D_COLS = [
    'inspection_key', 'immatriculation_norm', 'checkpoint_code',
    'checkpoint_libelle', 'zone_controle', 'tier',
    'valeur_controle', 'est_anomalie_critique',
    'commentaire_original', 'commentaire_clean',
    'observed_status_v2', 'penalty_applied_v2',
    'safety_grade_v2', 'decision_v2',
    'kw_advisory', 'kw_action_repair',
]

critique_with_advisory_comments = (
    df_staffim[mask_critique_advisory][AUDIT_D_COLS]
    .reset_index(drop=True)
)

print(f'Audit D — CRITIQUE with advisory-only comment: {len(critique_with_advisory_comments):,} rows')
print(f'  Distinct inspections : {critique_with_advisory_comments["inspection_key"].nunique():,}')

critique_with_advisory_comments.head(8)

Audit D — CRITIQUE with advisory-only comment: 6 rows
  Distinct inspections : 1


,inspection_key,immatriculation_norm,checkpoint_code,checkpoint_libelle,zone_controle,tier,valeur_controle,est_anomalie_critique,commentaire_original,commentaire_clean,observed_status_v2,penalty_applied_v2,safety_grade_v2,decision_v2,kw_advisory,kw_action_repair
0,7456TU157|20250811|6,7456TU157,autres_prestations_controle_filtre_a_air,Controle filtre a air,ENTRETIEN,T3_COSMETIC,NON,False,Vidange conseillé,vidange conseille,WORN,1.00,D,CRITIQUE,True,True
1,7456TU157|20250811|6,7456TU157,autres_prestations_controle_filtre_d_habitacle,Controle filtre d habitacle,ENTRETIEN,T3_COSMETIC,NON,False,Vidange conseillé,vidange conseille,WORN,1.00,D,CRITIQUE,True,True
2,7456TU157|20250811|6,7456TU157,autres_prestations_fonctionnement_climatisation,Fonctionnement climatisation,ENTRETIEN,T3_COSMETIC,OUI,None,Vidange conseillé,vidange conseille,NaN,NaN,D,CRITIQUE,True,True
3,7456TU157|20250811|6,7456TU157,autres_prestations_courroie_de_disribution,Courroie de disribution,ENTRETIEN,T2_CRITICAL,OUI,None,Vidange conseillé,vidange conseille,NaN,NaN,D,CRITIQUE,True,True
4,7456TU157|20250811|6,7456TU157,autres_prestations_controle_bougies_d_allumage,Controle bougies d allumage,ENTRETIEN,T2_NORMAL,OUI,None,Vidange conseillé,vidange conseille,NaN,NaN,D,CRITIQUE,True,True
5,7456TU157|20250811|6,7456TU157,autres_prestations_operation_d_entretien,Operation d entretien,ENTRETIEN,UNKNOWN_REVIEW,OUI,None,Vidange conseillé,vidange conseille,NaN,NaN,D,CRITIQUE,True,True


### Audit E — CRITIQUE / IMMOBILISE with Strong Defect Evidence in Comments

For the most severe VHS decisions we verify that comments **confirm** the severity. Strong defect words — `fuite`, `manque`, `a changer`, `frein`, `disque`, `plaquette`, `amortisseur` — in the comment corroborate the structured classification.

High counts here support keeping `VHS_BALANCED_V2` unchanged.

In [396]:
mask_critical_evidence = (
    df_staffim['decision_v2'].isin(['CRITIQUE', 'IMMOBILISE'])
    & df_staffim['has_comment']
    & _kw_strong_defect
)

AUDIT_E_COLS = [
    'inspection_key', 'immatriculation_norm', 'checkpoint_code',
    'checkpoint_libelle', 'zone_controle', 'tier',
    'valeur_controle', 'est_anomalie', 'est_anomalie_critique',
    'commentaire_original', 'commentaire_clean',
    'observed_status_v2', 'penalty_applied_v2',
    'safety_grade_v2', 'decision_v2',
    'kw_brake', 'kw_leak', 'kw_missing_level',
    'kw_change_required', 'kw_suspension_direction',
]

critical_comment_evidence = (
    df_staffim[mask_critical_evidence][AUDIT_E_COLS]
    .reset_index(drop=True)
)

print(f'Audit E — CRITIQUE/IMMOBILISE with strong defect comment: {len(critical_comment_evidence):,} rows')
print(f'  decision CRITIQUE   : {(critical_comment_evidence["decision_v2"]=="CRITIQUE").sum():,}')
print(f'  decision IMMOBILISE : {(critical_comment_evidence["decision_v2"]=="IMMOBILISE").sum():,}')
print(f'  kw_brake            : {critical_comment_evidence["kw_brake"].sum():,}')
print(f'  kw_leak             : {critical_comment_evidence["kw_leak"].sum():,}')
print(f'  kw_change_required  : {critical_comment_evidence["kw_change_required"].sum():,}')

critical_comment_evidence.head(8)

Audit E — CRITIQUE/IMMOBILISE with strong defect comment: 454 rows
  decision CRITIQUE   : 441
  decision IMMOBILISE : 13
  kw_brake            : 227
  kw_leak             : 149
  kw_change_required  : 323


,inspection_key,immatriculation_norm,checkpoint_code,checkpoint_libelle,zone_controle,tier,valeur_controle,est_anomalie,est_anomalie_critique,commentaire_original,commentaire_clean,observed_status_v2,penalty_applied_v2,safety_grade_v2,decision_v2,kw_brake,kw_leak,kw_missing_level,kw_change_required,kw_suspension_direction
0,1584TU107|20250811|5,1584TU107,autres_prestations_controle_filtre_a_air,Controle filtre a air,ENTRETIEN,T3_COSMETIC,NON,True,False,Ne dispose pas d'un climatiseur Courroie distribution avec traces d'huile Filtre à air à remplacer,ne dispose pas d'un climatiseur courroie distribution avec traces d'huile filtre a air a remplacer,WORN,1.00,D,CRITIQUE,False,False,False,True,False
1,1584TU107|20250811|5,1584TU107,autres_prestations_controle_filtre_d_habitacle,Controle filtre d habitacle,ENTRETIEN,T3_COSMETIC,NON,True,False,Ne dispose pas d'un climatiseur Courroie distribution avec traces d'huile Filtre à air à remplacer,ne dispose pas d'un climatiseur courroie distribution avec traces d'huile filtre a air a remplacer,WORN,1.00,D,CRITIQUE,False,False,False,True,False
2,1584TU107|20250811|5,1584TU107,autres_prestations_courroie_de_disribution,Courroie de disribution,ENTRETIEN,T2_CRITICAL,NON,True,False,Ne dispose pas d'un climatiseur Courroie distribution avec traces d'huile Filtre à air à remplacer,ne dispose pas d'un climatiseur courroie distribution avec traces d'huile filtre a air a remplacer,WORN,5.00,D,CRITIQUE,False,False,False,True,False
3,1584TU107|20250811|5,1584TU107,autres_prestations_fonctionnement_climatisation,Fonctionnement climatisation,ENTRETIEN,T3_COSMETIC,NON,True,False,Ne dispose pas d'un climatiseur Courroie distribution avec traces d'huile Filtre à air à remplacer,ne dispose pas d'un climatiseur courroie distribution avec traces d'huile filtre a air a remplacer,WORN,1.00,D,CRITIQUE,False,False,False,True,False
4,1584TU107|20250811|5,1584TU107,sous_le_capot_controle_du_niveau_du_liquide_de_refroidi,Controle du niveau du liquide de refroidi,SOUS_CAPOT,T2_CRITICAL,Contrôle non OK,True,True,Fuite eau radiateur Vase d'eau invisible Manque huile moteur,fuite eau radiateur vase d'eau invisible manque huile moteur,BROKEN,12.00,D,CRITIQUE,False,True,True,False,False
5,1584TU107|20250811|5,1584TU107,sous_le_capot_controle_du_niveau_huile_moteur,Controle du niveau huile moteur,SOUS_CAPOT,T2_CRITICAL,Contrôle non OK,True,True,Fuite eau radiateur Vase d'eau invisible Manque huile moteur,fuite eau radiateur vase d'eau invisible manque huile moteur,BROKEN,12.00,D,CRITIQUE,False,True,True,False,False
6,1584TU107|20250811|5,1584TU107,sous_le_capot_controle_durits_de_radiateur,Controle durits de radiateur,SOUS_CAPOT,T2_NORMAL,Contrôle non OK,True,True,Fuite eau radiateur Vase d'eau invisible Manque huile moteur,fuite eau radiateur vase d'eau invisible manque huile moteur,BROKEN,8.00,D,CRITIQUE,False,True,True,False,False
7,1584TU107|20250811|5,1584TU107,sous_le_vehicule_controle_des_plaquettes_de_freins_av,Controle des plaquettes de freins av,SOUS_VEHICULE,T1_VITAL,Contrôle non OK,True,True,Corrosion partout Fuite huile moteur et eau Jeu direction Plaquette et disque de frein avant à remplacer,corrosion partout fuite huile moteur et eau jeu direction plaquette et disque de frein avant a remplacer,BROKEN,25.00,D,CRITIQUE,True,True,False,True,True


In [397]:
# Consolidated audit summary
audit_summary = pd.DataFrame([
    {'audit': 'A', 'description': 'OK value + defect keyword in comment',
     'n_rows': len(comments_with_ok_value)},
    {'audit': 'B', 'description': 'PROPOSITION FAITE (all)',
     'n_rows': len(proposition_faite_comment_context)},
    {'audit': 'B-advisory', 'description': 'PROPOSITION FAITE — advisory comment only',
     'n_rows': n_prop_adv},
    {'audit': 'B-strong',   'description': 'PROPOSITION FAITE — strong defect in comment',
     'n_rows': n_prop_strong},
    {'audit': 'C', 'description': 'Intervention conseillee (all)',
     'n_rows': len(intervention_conseillee_comment_context)},
    {'audit': 'C-worn-strong', 'description': 'Intervention conseillee — WORN_STRONG',
     'n_rows': n_worn_strong},
    {'audit': 'D', 'description': 'CRITIQUE + advisory-only comment',
     'n_rows': len(critique_with_advisory_comments)},
    {'audit': 'E', 'description': 'CRITIQUE/IMMOBILISE + strong defect comment',
     'n_rows': len(critical_comment_evidence)},
])

print('=== Audit Summary ===')
print(audit_summary.to_string(index=False))

=== Audit Summary ===
        audit                                  description  n_rows
            A         OK value + defect keyword in comment    2119
            B                      PROPOSITION FAITE (all)     105
   B-advisory    PROPOSITION FAITE — advisory comment only       2
     B-strong PROPOSITION FAITE — strong defect in comment      27
            C                Intervention conseillee (all)     575
C-worn-strong        Intervention conseillee — WORN_STRONG     269
            D             CRITIQUE + advisory-only comment       6
            E  CRITIQUE/IMMOBILISE + strong defect comment     454


---
## 7. Ambiguity Score — Comment Evidence Level

We assign an exploratory `comment_evidence_level` to every row based on keyword presence in `commentaire_clean`. This is a **diagnostic label**, not a VHS score.

| Level | Assigned when | Meaning |
|-------|--------------|---------|
| `STRONG_DEFECT` | Comment contains brake, suspension, leak, change-required, or missing-level keywords | Strong corroboration of a physical defect |
| `ADVISORY` | Comment contains advisory keywords only (no strong defect) | Inspector suggested an action; defect not confirmed |
| `WEAK_OR_UNCLEAR` | Comment exists but no keyword group matched | Comment present but too vague or abbreviated to classify |
| `NO_COMMENT` | No comment at all | No free-text evidence available |

> This four-level scale intentionally has no numeric value. It is used only for cross-tab analysis and human review — never as a scoring input.

In [398]:
def get_comment_evidence_level(row) -> str:
    """Rule-based comment evidence level. NOT a VHS score."""
    if not row['has_comment']:
        return 'NO_COMMENT'
    # Strong defect: direct safety / mechanical evidence
    if (
        row['kw_brake']
        or row['kw_suspension_direction']
        or row['kw_leak']
        or row['kw_change_required']
        or row['kw_missing_level']
    ):
        return 'STRONG_DEFECT'
    # Advisory: soft suggestion language only
    if row['kw_advisory']:
        return 'ADVISORY'
    # Comment present but no matching keyword
    return 'WEAK_OR_UNCLEAR'


df_staffim['comment_evidence_level'] = df_staffim.apply(
    get_comment_evidence_level, axis=1
)

EVIDENCE_ORDER = ['STRONG_DEFECT', 'ADVISORY', 'WEAK_OR_UNCLEAR', 'NO_COMMENT']

ev_dist = df_staffim['comment_evidence_level'].value_counts().reindex(EVIDENCE_ORDER, fill_value=0)

print('=== comment_evidence_level distribution ===')
print(f'  {"Level":<20}  {"Count":>8}  {"% rows":>8}')
print('  ' + '-' * 42)
for level, cnt in ev_dist.items():
    pct = 100 * cnt / len(df_staffim)
    print(f'  {level:<20}  {cnt:>8,}  {pct:>7.1f}%')

=== comment_evidence_level distribution ===
  Level                    Count    % rows
  ------------------------------------------
  STRONG_DEFECT            2,386     19.4%
  ADVISORY                    36      0.3%
  WEAK_OR_UNCLEAR          1,548     12.6%
  NO_COMMENT               8,328     67.7%


In [399]:
# Evidence level among rows WITH a comment, broken down by decision_v2
df_ev = df_staffim[df_staffim['has_comment']].copy()

ev_by_decision = (
    df_ev.groupby(['decision_v2', 'comment_evidence_level'], dropna=False)
    .size()
    .unstack(fill_value=0)
    .reindex(columns=[c for c in EVIDENCE_ORDER if c in df_ev['comment_evidence_level'].unique()])
)

print('=== Evidence level by decision_v2 (commented rows only) ===')
print(ev_by_decision.to_string())

# Row-percentage version
ev_by_decision_pct = ev_by_decision.div(ev_by_decision.sum(axis=1), axis=0).mul(100).round(1)
print('\n=== Same table as row % ===')
print(ev_by_decision_pct.to_string())

=== Evidence level by decision_v2 (commented rows only) ===
comment_evidence_level  STRONG_DEFECT  ADVISORY  WEAK_OR_UNCLEAR
decision_v2                                                     
CRITIQUE                          441         6              292
DEGRADE                          1617        18              765
IMMOBILISE                         13         0               24
OK                                315        12              467

=== Same table as row % ===
comment_evidence_level  STRONG_DEFECT  ADVISORY  WEAK_OR_UNCLEAR
decision_v2                                                     
CRITIQUE                        59.70      0.80            39.50
DEGRADE                         67.40      0.80            31.90
IMMOBILISE                      35.10      0.00            64.90
OK                              39.70      1.50            58.80


In [400]:
# Evidence level by observed_status_v2 (scored rows only)
df_ev_scored = df_staffim[df_staffim['has_comment'] & df_staffim['observed_status_v2'].notna()].copy()

ev_by_status = (
    df_ev_scored.groupby(['observed_status_v2', 'comment_evidence_level'], dropna=False)
    .size()
    .unstack(fill_value=0)
    .reindex(columns=[c for c in EVIDENCE_ORDER if c in df_ev_scored['comment_evidence_level'].unique()])
)

print('=== Evidence level by observed_status_v2 (commented + scored rows) ===')
print(ev_by_status.to_string())

ev_by_status_pct = ev_by_status.div(ev_by_status.sum(axis=1), axis=0).mul(100).round(1)
print('\n=== Same table as row % ===')
print(ev_by_status_pct.to_string())

=== Evidence level by observed_status_v2 (commented + scored rows) ===
comment_evidence_level  STRONG_DEFECT  ADVISORY  WEAK_OR_UNCLEAR
observed_status_v2                                              
BROKEN                             84         0               32
REPAIRED                            5         0                0
WORN                              191        11              194
WORN_STRONG                       182         0               22

=== Same table as row % ===
comment_evidence_level  STRONG_DEFECT  ADVISORY  WEAK_OR_UNCLEAR
observed_status_v2                                              
BROKEN                          72.40      0.00            27.60
REPAIRED                       100.00      0.00             0.00
WORN                            48.20      2.80            49.00
WORN_STRONG                     89.20      0.00            10.80


---
## 8. Cross-Tab Analysis

We cross-tabulate `comment_evidence_level` against five VHS dimensions. Both **counts** and **row-percentages** are shown so the distribution pattern is visible regardless of group size.

**How to read these tables:**
- A high `STRONG_DEFECT` share among BROKEN / CRITIQUE rows supports keeping V2 unchanged.
- A high `ADVISORY` share among BROKEN / CRITIQUE rows is a signal that some rows may be over-scored — potential motivation for V3.
- A high `NO_COMMENT` share means comments are too sparse to draw conclusions.

In [401]:
def show_crosstab(df, index_col, title=None, order=None):
    """Print count and row-% cross-tab of index_col vs comment_evidence_level."""
    order = order or EVIDENCE_ORDER
    ct = pd.crosstab(df[index_col], df['comment_evidence_level'])
    # Keep only columns that exist and respect the desired order
    cols = [c for c in order if c in ct.columns]
    ct  = ct[cols]
    pct = ct.div(ct.sum(axis=1), axis=0).mul(100).round(1)

    label = title or index_col
    print(f'=== {label} x comment_evidence_level (counts) ===')
    print(ct.to_string())
    print(f'\n=== {label} x comment_evidence_level (row %) ===')
    print(pct.to_string())
    print()
    return ct, pct

In [402]:
ct_valeur, pct_valeur = show_crosstab(
    df_staffim,
    'valeur_controle',
    title='valeur_controle',
)

=== valeur_controle x comment_evidence_level (counts) ===
comment_evidence_level                        STRONG_DEFECT  ADVISORY  WEAK_OR_UNCLEAR  NO_COMMENT
valeur_controle                                                                                   
Bon                                                     207         0              356        1710
Contrôle OK                                            1553         8              784        4770
Contrôle non OK                                          94         0               30         180
Défectueux                                               47         0               99         116
Intervention conseillée                                 296         4              136         139
NON                                                      44         7               34          64
OUI                                                     116        15               76        1294
PROPOSITION FAITE                                  

In [403]:
ct_status, pct_status = show_crosstab(
    df_staffim[df_staffim['observed_status_v2'].notna()],
    'observed_status_v2',
    title='observed_status_v2',
)

=== observed_status_v2 x comment_evidence_level (counts) ===
comment_evidence_level  STRONG_DEFECT  ADVISORY  WEAK_OR_UNCLEAR  NO_COMMENT
observed_status_v2                                                          
BROKEN                             84         0               32         112
REPAIRED                            5         0                0           7
WORN                              191        11              194         281
WORN_STRONG                       182         0               22          65

=== observed_status_v2 x comment_evidence_level (row %) ===
comment_evidence_level  STRONG_DEFECT  ADVISORY  WEAK_OR_UNCLEAR  NO_COMMENT
observed_status_v2                                                          
BROKEN                          36.80      0.00            14.00       49.10
REPAIRED                        41.70      0.00             0.00       58.30
WORN                            28.20      1.60            28.70       41.50
WORN_STRONG                    

In [404]:
ct_grade, pct_grade = show_crosstab(
    df_staffim[df_staffim['safety_grade_v2'].notna()],
    'safety_grade_v2',
    title='safety_grade_v2',
)

=== safety_grade_v2 x comment_evidence_level (counts) ===
comment_evidence_level  STRONG_DEFECT  ADVISORY  WEAK_OR_UNCLEAR  NO_COMMENT
safety_grade_v2                                                             
A                                 243        12              482        3348
B                                 129         0               45         256
C                                1573        18              729        3270
D                                 441         6              292        1454

=== safety_grade_v2 x comment_evidence_level (row %) ===
comment_evidence_level  STRONG_DEFECT  ADVISORY  WEAK_OR_UNCLEAR  NO_COMMENT
safety_grade_v2                                                             
A                                5.90      0.30            11.80       82.00
B                               30.00      0.00            10.50       59.50
C                               28.10      0.30            13.00       58.50
D                               20.10

In [405]:
ct_decision, pct_decision = show_crosstab(
    df_staffim[df_staffim['decision_v2'].notna()],
    'decision_v2',
    title='decision_v2',
)

=== decision_v2 x comment_evidence_level (counts) ===
comment_evidence_level  STRONG_DEFECT  ADVISORY  WEAK_OR_UNCLEAR  NO_COMMENT
decision_v2                                                                 
CRITIQUE                          441         6              292        1454
DEGRADE                          1617        18              765        3319
IMMOBILISE                         13         0               24           6
OK                                315        12              467        3549

=== decision_v2 x comment_evidence_level (row %) ===
comment_evidence_level  STRONG_DEFECT  ADVISORY  WEAK_OR_UNCLEAR  NO_COMMENT
decision_v2                                                                 
CRITIQUE                        20.10      0.30            13.30       66.30
DEGRADE                         28.30      0.30            13.40       58.00
IMMOBILISE                      30.20      0.00            55.80       14.00
OK                               7.30      0.

In [406]:
ct_tier, pct_tier = show_crosstab(
    df_staffim,
    'tier',
    title='tier',
)

=== tier x comment_evidence_level (counts) ===
comment_evidence_level  STRONG_DEFECT  ADVISORY  WEAK_OR_UNCLEAR  NO_COMMENT
tier                                                                        
T1_IMPORTANT                      266         0              148         730
T1_VITAL                          673         2              158        1169
T2_CRITICAL                       587        12              264        1711
T2_NORMAL                         326         6              479        2049
T3_COSMETIC                       114        12              113         905
UNKNOWN_REVIEW                    420         4              386        1764

=== tier x comment_evidence_level (row %) ===
comment_evidence_level  STRONG_DEFECT  ADVISORY  WEAK_OR_UNCLEAR  NO_COMMENT
tier                                                                        
T1_IMPORTANT                    23.30      0.00            12.90       63.80
T1_VITAL                        33.60      0.10            

In [407]:
# Compact summary: for each decision, what % of commented rows are STRONG_DEFECT vs ADVISORY?
decisions_of_interest = ['OK', 'DEGRADE', 'CRITIQUE', 'IMMOBILISE']

print('=== STRONG_DEFECT vs ADVISORY share by decision_v2 (commented rows only) ===')
print(f'  {"decision_v2":<15}  {"STRONG_DEFECT %":>16}  {"ADVISORY %":>12}  {"WEAK/UNCLEAR %":>15}  {"n rows":>7}')
print('  ' + '-' * 72)

df_dec_ev = df_staffim[df_staffim['has_comment'] & df_staffim['decision_v2'].notna()]

for dec in decisions_of_interest:
    sub = df_dec_ev[df_dec_ev['decision_v2'] == dec]
    if sub.empty:
        continue
    n = len(sub)
    p_strong = 100 * (sub['comment_evidence_level'] == 'STRONG_DEFECT').sum() / n
    p_adv    = 100 * (sub['comment_evidence_level'] == 'ADVISORY').sum() / n
    p_weak   = 100 * (sub['comment_evidence_level'] == 'WEAK_OR_UNCLEAR').sum() / n
    print(f'  {dec:<15}  {p_strong:>15.1f}%  {p_adv:>11.1f}%  {p_weak:>14.1f}%  {n:>7,}')

=== STRONG_DEFECT vs ADVISORY share by decision_v2 (commented rows only) ===
  decision_v2       STRONG_DEFECT %    ADVISORY %   WEAK/UNCLEAR %   n rows
  ------------------------------------------------------------------------
  OK                          39.7%          1.5%            58.8%      794
  DEGRADE                     67.4%          0.8%            31.9%    2,400
  CRITIQUE                    59.7%          0.8%            39.5%      739
  IMMOBILISE                  35.1%          0.0%            64.9%       37


---
## 9. Representative Examples

We display up to 4 rows per scenario to support qualitative interpretation alongside the quantitative findings above.

| Scenario | Description |
|----------|-------------|
| S1 | OK structured value but comment suggests a defect |
| S2 | `PROPOSITION FAITE` with advisory comment only |
| S3 | `PROPOSITION FAITE` with strong defect comment |
| S4 | `Intervention conseillee` with strong defect comment |
| S5 | CRITIQUE decision — comment confirms severity |
| S6 | CRITIQUE decision — comment is advisory only |

Each row shows the full structured context alongside the original inspector comment and the `comment_evidence_level` assigned in Section 7.

In [408]:
EXAMPLE_COLS = [
    'inspection_key',
    'immatriculation_norm',
    'checkpoint_libelle',
    'tier',
    'valeur_controle',
    'est_anomalie',
    'est_anomalie_critique',
    'commentaire_original',
    'observed_status_v2',
    'penalty_applied_v2',
    'safety_grade_v2',
    'decision_v2',
    'comment_evidence_level',
]


def print_examples(df_sub, label, n=4):
    """Pretty-print up to n rows for a given scenario."""
    sample = df_sub.head(n)
    print(f'\n{'='*72}')
    print(f' {label}  ({min(n, len(df_sub))} of {len(df_sub):,} matching rows)')
    print(f'{'='*72}')
    if sample.empty:
        print('  (no matching rows)')
        return
    for i, (_, row) in enumerate(sample[EXAMPLE_COLS].iterrows(), 1):
        print(f'\n  Example {i}:')
        for col in EXAMPLE_COLS:
            val = str(row[col]) if row[col] is not None else '—'
            print(f'    {col:<28}: {val[:100]}')
        print('  ' + '-'*68)

In [409]:
# S1: OK structured value but defect keyword in comment
s1 = df_staffim[
    df_staffim['valeur_controle'].isin(OK_VALUES)
    & df_staffim['has_comment']
    & _kw_strong_defect
][EXAMPLE_COLS]

print_examples(s1, 'S1 — OK/Bon value, strong defect keyword in comment')


 S1 — OK/Bon value, strong defect keyword in comment  (4 of 1,876 matching rows)

  Example 1:
    inspection_key              : 9432TU156|20250901|22
    immatriculation_norm        : 9432TU156
    checkpoint_libelle          : Controle batterie etat fixation et charge
    tier                        : T2_CRITICAL
    valeur_controle             : Contrôle OK
    est_anomalie                : False
    est_anomalie_critique       : —
    commentaire_original        : Bouchon vase d'eau à remplacer
    observed_status_v2          : nan
    penalty_applied_v2          : nan
    safety_grade_v2             : B
    decision_v2                 : DEGRADE
    comment_evidence_level      : STRONG_DEFECT
  --------------------------------------------------------------------

  Example 2:
    inspection_key              : 3475TU146|20250918|56
    immatriculation_norm        : 3475TU146
    checkpoint_libelle          : Controle fonctionnement des feux ecl 1
    tier                        : U

In [410]:
# S2: PROPOSITION FAITE with advisory comment only (no strong defect)
# Re-filter from df_staffim so that comment_evidence_level is available.
_prop_strong_df = (
    df_staffim['kw_brake']
    | df_staffim['kw_leak']
    | df_staffim['kw_change_required']
    | df_staffim['kw_missing_level']
    | df_staffim['kw_engine_fluids']
    | df_staffim['kw_suspension_direction']
)
s2 = df_staffim[
    mask_prop
    & df_staffim['has_comment']
    & df_staffim['kw_advisory']
    & ~_prop_strong_df
][EXAMPLE_COLS].reset_index(drop=True)

print_examples(s2, 'S2 — PROPOSITION FAITE, advisory comment only')


 S2 — PROPOSITION FAITE, advisory comment only  (2 of 2 matching rows)

  Example 1:
    inspection_key              : 5620TU154|20251231|160
    immatriculation_norm        : 5620TU154
    checkpoint_libelle          : Courroie de disribution
    tier                        : T2_CRITICAL
    valeur_controle             : PROPOSITION FAITE
    est_anomalie                : True
    est_anomalie_critique       : False
    commentaire_original        : vidange conseillé
    observed_status_v2          : WORN
    penalty_applied_v2          : 5.0
    safety_grade_v2             : C
    decision_v2                 : DEGRADE
    comment_evidence_level      : ADVISORY
  --------------------------------------------------------------------

  Example 2:
    inspection_key              : 3258TU205|20260504|273
    immatriculation_norm        : 3258TU205
    checkpoint_libelle          : Operation d entretien
    tier                        : UNKNOWN_REVIEW
    valeur_controle             : PRO

In [411]:
# S3: PROPOSITION FAITE with strong defect comment
s3 = df_staffim[
    mask_prop
    & df_staffim['has_comment']
    & _prop_strong_df
][EXAMPLE_COLS].reset_index(drop=True)

print_examples(s3, 'S3 — PROPOSITION FAITE, strong defect in comment')


 S3 — PROPOSITION FAITE, strong defect in comment  (4 of 27 matching rows)

  Example 1:
    inspection_key              : 1296TU227|20260213|199
    immatriculation_norm        : 1296TU227
    checkpoint_libelle          : Controle bougies d allumage
    tier                        : T2_NORMAL
    valeur_controle             : PROPOSITION FAITE
    est_anomalie                : True
    est_anomalie_critique       : False
    commentaire_original        : Filtre a air a remplacer les Bougies sont remplacer
    observed_status_v2          : WORN
    penalty_applied_v2          : 3.0
    safety_grade_v2             : A
    decision_v2                 : OK
    comment_evidence_level      : STRONG_DEFECT
  --------------------------------------------------------------------

  Example 2:
    inspection_key              : 3258TU237|20251017|81
    immatriculation_norm        : 3258TU237
    checkpoint_libelle          : Controle filtre a air
    tier                        : T3_COSMETIC
 

In [412]:
# S4: Intervention conseillee with strong defect comment
# Re-filter from df_staffim so that comment_evidence_level is available.
_interv_strong_df = (
    df_staffim['kw_brake']
    | df_staffim['kw_suspension_direction']
    | df_staffim['kw_leak']
    | df_staffim['kw_change_required']
    | df_staffim['kw_engine_fluids']
)
s4 = df_staffim[
    mask_interv
    & df_staffim['has_comment']
    & _interv_strong_df
][EXAMPLE_COLS].reset_index(drop=True)

print_examples(s4, 'S4 — Intervention conseillee, strong defect in comment')


 S4 — Intervention conseillee, strong defect in comment  (4 of 354 matching rows)

  Example 1:
    inspection_key              : 1089TU113|20260330|236
    immatriculation_norm        : 1089TU113
    checkpoint_libelle          : Controle batterie etat fixation et charge
    tier                        : T2_CRITICAL
    valeur_controle             : Intervention conseillée
    est_anomalie                : True
    est_anomalie_critique       : False
    commentaire_original        : DEMARREUR A REPARER / REVISION OU REMPLACEMENT DE MOTEUR
    observed_status_v2          : WORN
    penalty_applied_v2          : 5.0
    safety_grade_v2             : C
    decision_v2                 : DEGRADE
    comment_evidence_level      : STRONG_DEFECT
  --------------------------------------------------------------------

  Example 2:
    inspection_key              : 1089TU113|20260330|236
    immatriculation_norm        : 1089TU113
    checkpoint_libelle          : Controle du niveau huile mote

In [413]:
# S5: CRITIQUE decision — comment confirms severity
# Re-filter from df_staffim so that comment_evidence_level is available.
s5 = df_staffim[
    mask_critical_evidence
    & (df_staffim['decision_v2'] == 'CRITIQUE')
][EXAMPLE_COLS].reset_index(drop=True)
print_examples(s5, 'S5 — CRITIQUE decision, strong defect comment evidence')

# S6: CRITIQUE decision — advisory-only comment
s6 = df_staffim[mask_critique_advisory][EXAMPLE_COLS].reset_index(drop=True)
print_examples(s6, 'S6 — CRITIQUE decision, advisory-only comment')


 S5 — CRITIQUE decision, strong defect comment evidence  (4 of 441 matching rows)

  Example 1:
    inspection_key              : 1584TU107|20250811|5
    immatriculation_norm        : 1584TU107
    checkpoint_libelle          : Controle filtre a air
    tier                        : T3_COSMETIC
    valeur_controle             : NON
    est_anomalie                : True
    est_anomalie_critique       : False
    commentaire_original        : Ne dispose pas d'un climatiseur Courroie distribution avec traces d'huile Filtre à air à remplacer
    observed_status_v2          : WORN
    penalty_applied_v2          : 1.0
    safety_grade_v2             : D
    decision_v2                 : CRITIQUE
    comment_evidence_level      : STRONG_DEFECT
  --------------------------------------------------------------------

  Example 2:
    inspection_key              : 1584TU107|20250811|5
    immatriculation_norm        : 1584TU107
    checkpoint_libelle          : Controle filtre d habitacle
  

In [414]:
# Consolidate all examples into one export dataframe
scenario_frames = [
    (s1, 'S1_OK_WITH_DEFECT_COMMENT'),
    (s2, 'S2_PROPOSITION_ADVISORY_ONLY'),
    (s3, 'S3_PROPOSITION_STRONG_DEFECT'),
    (s4, 'S4_INTERVENTION_STRONG_DEFECT'),
    (s5, 'S5_CRITIQUE_DEFECT_EVIDENCE'),
    (s6, 'S6_CRITIQUE_ADVISORY_COMMENT'),
]

parts = []
for df_sc, label in scenario_frames:
    chunk = df_sc.head(4).copy()
    chunk.insert(0, 'scenario', label)
    parts.append(chunk)

staffim_comment_examples = pd.concat(parts, ignore_index=True)
print(f'Total example rows consolidated: {len(staffim_comment_examples)}')
staffim_comment_examples.head(24)

Total example rows consolidated: 22


,scenario,inspection_key,immatriculation_norm,checkpoint_libelle,tier,valeur_controle,est_anomalie,est_anomalie_critique,commentaire_original,observed_status_v2,penalty_applied_v2,safety_grade_v2,decision_v2,comment_evidence_level
0,S1_OK_WITH_DEFECT_COMMENT,9432TU156|20250901|22,9432TU156,Controle batterie etat fixation et charge,T2_CRITICAL,Contrôle OK,False,None,Bouchon vase d'eau à remplacer,NaN,NaN,B,DEGRADE,STRONG_DEFECT
1,S1_OK_WITH_DEFECT_COMMENT,3475TU146|20250918|56,3475TU146,Controle fonctionnement des feux ecl 1,UNKNOWN_REVIEW,Contrôle OK,False,None,Manque liquide essuie glace 2 veilleuse av ne fonctionne pas,NaN,NaN,C,DEGRADE,STRONG_DEFECT
2,S1_OK_WITH_DEFECT_COMMENT,8613TU256|20260331|237,8613TU256,Controle ligne d echappement et fixati,T2_CRITICAL,Contrôle OK,False,None,02 AMORTISSEURS AR + 02 ANTICHOCS ARRIERE A REMPLACER,NaN,NaN,C,DEGRADE,STRONG_DEFECT
3,S1_OK_WITH_DEFECT_COMMENT,8041TU243|20260514|283,8041TU243,Vitres et pare brise,T2_NORMAL,Bon,False,None,PNEUS AVANT A REMPLACER,NaN,NaN,C,DEGRADE,STRONG_DEFECT
4,S2_PROPOSITION_ADVISORY_ONLY,5620TU154|20251231|160,5620TU154,Courroie de disribution,T2_CRITICAL,PROPOSITION FAITE,True,False,vidange conseillé,WORN,5.00,C,DEGRADE,ADVISORY
5,S2_PROPOSITION_ADVISORY_ONLY,3258TU205|20260504|273,3258TU205,Operation d entretien,UNKNOWN_REVIEW,PROPOSITION FAITE,True,False,VIDANGE CONSEILLE,NaN,NaN,A,OK,ADVISORY
6,S3_PROPOSITION_STRONG_DEFECT,1296TU227|20260213|199,1296TU227,Controle bougies d allumage,T2_NORMAL,PROPOSITION FAITE,True,False,Filtre a air a remplacer les Bougies sont remplacer,WORN,3.00,A,OK,STRONG_DEFECT
7,S3_PROPOSITION_STRONG_DEFECT,3258TU237|20251017|81,3258TU237,Controle filtre a air,T3_COSMETIC,PROPOSITION FAITE,True,False,FILTRE A AIR A REMPLACER,WORN,1.00,A,OK,STRONG_DEFECT
8,S3_PROPOSITION_STRONG_DEFECT,3613TU166|20260326|228,3613TU166,Balais essuie glace,T2_NORMAL,Proposition faite,True,False,pneu arriere gauche a remplacer essuie glace arriere a remplacer,WORN,3.00,C,DEGRADE,STRONG_DEFECT
9,S3_PROPOSITION_STRONG_DEFECT,3613TU166|20260326|228,3613TU166,Pneus arriere,T1_IMPORTANT,Proposition faite,True,True,pneu arriere gauche a remplacer essuie glace arriere a remplacer,BROKEN,15.00,C,DEGRADE,STRONG_DEFECT


---
## Audit: The `NON` Value in STAFFIM

`NON` (and its case variants) is the most ambiguous `valeur_controle` value.
It can mean:

- A **negative answer** to a yes/no check ("Is the tire pressure correct? NON")
- A **non-conformity** flag (the item does not conform to the norm)
- An **action not performed** (e.g. the inspector could not test the item)
- A **real defect** (equivalent to BROKEN in severity)

`NON` therefore does **not** automatically map to `BROKEN`.
This section audits how `NON` behaves across zones, checkpoints, anomaly flags,
VHS statuses, and comment content, to support a future mapping decision.

> **Read-only throughout.** No VHS is recomputed. No table is written.

### NON-1 — Filter: Build `df_non`

Subset of `df_staffim` where `valeur_controle` equals `"NON"` (case-insensitive).
We retain all 27 analytical columns so every downstream cell can access them.

In [415]:
NON_COLS = [
    'inspection_key', 'immatriculation_norm',
    'checkpoint_code', 'libelle_checkpoint', 'zone_controle',
    'valeur_controle', 'tier', 'is_vital', 'is_important',
    'est_anomalie_critique', 'est_anomalie_securite',
    'commentaire_original', 'commentaire_clean',
    'has_comment', 'comment_length', 'detected_keywords',
    'comment_evidence_level',
    'observed_status_v2', 'decision_v2', 'safety_grade_v2',
    'penalty_v2', 'penalty_worn', 'penalty_broken',
    'is_hard_cap_trigger',
]
# Keep only columns that exist in df_staffim
NON_COLS = [c for c in NON_COLS if c in df_staffim.columns]

mask_non = df_staffim['valeur_controle'].str.upper().str.strip() == 'NON'
df_non = df_staffim.loc[mask_non, NON_COLS].copy().reset_index(drop=True)

print(f'df_non: {len(df_non):,} rows  |  {df_non["inspection_key"].nunique():,} inspections  |  {df_non["immatriculation_norm"].nunique():,} vehicles')
print(f'As % of full df_staffim: {100*len(df_non)/len(df_staffim):.1f}%')
print(f'Columns retained: {list(df_non.columns)}')

df_non: 149 rows  |  73 inspections  |  73 vehicles
As % of full df_staffim: 1.2%
Columns retained: ['inspection_key', 'immatriculation_norm', 'checkpoint_code', 'zone_controle', 'valeur_controle', 'tier', 'is_vital', 'is_important', 'est_anomalie_critique', 'commentaire_original', 'commentaire_clean', 'has_comment', 'comment_length', 'detected_keywords', 'comment_evidence_level', 'observed_status_v2', 'decision_v2', 'safety_grade_v2', 'penalty_worn', 'penalty_broken', 'is_hard_cap_trigger']


### NON-2 — Profile: How does `NON` distribute?

Three slices: global counts, by zone/tier, by checkpoint and anomaly flags.

In [416]:
# Global counts and anomaly breakdown
n_non = len(df_non)
n_non_comment  = int(df_non['has_comment'].sum())
n_non_critique = int(df_non['est_anomalie_critique'].sum()) if 'est_anomalie_critique' in df_non.columns else 0
n_non_securite = int(df_non['est_anomalie_securite'].sum()) if 'est_anomalie_securite' in df_non.columns else 0
n_non_hardcap  = int(df_non['is_hard_cap_trigger'].sum()) if 'is_hard_cap_trigger' in df_non.columns else 0
n_non_broken   = int((df_non['observed_status_v2'] == 'BROKEN').sum())
n_non_worn_s   = int((df_non['observed_status_v2'] == 'WORN_STRONG').sum())
n_non_worn     = int((df_non['observed_status_v2'] == 'WORN').sum())
n_non_ok       = int((df_non['observed_status_v2'] == 'OK').sum())

print('=== Global NON profile ===')
print(f'Total NON rows           : {n_non:>6,}')
print(f'  with comment           : {n_non_comment:>6,}  ({100*n_non_comment/n_non:.1f}%)')
print(f'  est_anomalie_critique  : {n_non_critique:>6,}  ({100*n_non_critique/n_non:.1f}%)')
if 'est_anomalie_securite' in df_non.columns:
    print(f'  est_anomalie_securite  : {n_non_securite:>6,}  ({100*n_non_securite/n_non:.1f}%)')
print(f'  is_hard_cap_trigger    : {n_non_hardcap:>6,}  ({100*n_non_hardcap/n_non:.1f}%)')
print()
print('=== NON by observed_status_v2 ===')
print(df_non['observed_status_v2'].value_counts().to_string())
print()
print('=== NON by decision_v2 ===')
print(df_non['decision_v2'].value_counts().to_string())
print()
print('=== NON by comment_evidence_level ===')
print(df_non['comment_evidence_level'].value_counts().to_string())

=== Global NON profile ===
Total NON rows           :    149
  with comment           :     85  (57.0%)
  est_anomalie_critique  :      0  (0.0%)
  is_hard_cap_trigger    :      0  (0.0%)

=== NON by observed_status_v2 ===
observed_status_v2
WORN    128

=== NON by decision_v2 ===
decision_v2
DEGRADE       84
OK            36
CRITIQUE      27
IMMOBILISE     2

=== NON by comment_evidence_level ===
comment_evidence_level
NO_COMMENT         64
STRONG_DEFECT      44
WEAK_OR_UNCLEAR    34
ADVISORY            7


In [417]:
# NON by zone_controle and tier
print('=== NON by zone_controle (top 15) ===')
print(df_non['zone_controle'].value_counts().head(15).to_string())
print()
print('=== NON by tier ===')
print(df_non['tier'].value_counts().to_string())

=== NON by zone_controle (top 15) ===
zone_controle
ENTRETIEN    149

=== NON by tier ===
tier
T3_COSMETIC       106
UNKNOWN_REVIEW     21
T2_CRITICAL        13
T2_NORMAL           9


In [418]:
# NON by checkpoint code (top 20)
print('=== Top 20 checkpoints where valeur_controle == NON ===')
_grp_cols = [c for c in ['checkpoint_code', 'libelle_checkpoint'] if c in df_non.columns]
tmp = df_non.groupby(_grp_cols).size().sort_values(ascending=False).head(20)
print(tmp.to_string())

=== Top 20 checkpoints where valeur_controle == NON ===
checkpoint_code
autres_prestations_controle_filtre_d_habitacle     41
autres_prestations_controle_filtre_a_air           34
autres_prestations_fonctionnement_climatisation    31
autres_prestations_operation_d_entretien           21
autres_prestations_courroie_de_disribution         13
autres_prestations_controle_bougies_d_allumage      9


### NON-3 — Checkpoint Deep-Dive

For each checkpoint that has at least one `NON` row, compute 14 aggregate metrics:
count, comment rate, anomaly flags, hard-cap triggers, and VHS status breakdown.

In [419]:
_grp3 = [c for c in ['checkpoint_code', 'libelle_checkpoint', 'tier', 'zone_controle'] if c in df_non.columns]
non_by_checkpoint = df_non.groupby(_grp3, as_index=False).agg(
    n_non=('inspection_key', 'count'),
    n_with_comment=('has_comment', 'sum'),
    n_critique=('est_anomalie_critique', 'sum') if 'est_anomalie_critique' in df_non.columns else ('inspection_key', lambda x: 0),
    n_hard_cap=('is_hard_cap_trigger', 'sum') if 'is_hard_cap_trigger' in df_non.columns else ('inspection_key', lambda x: 0),
    n_broken=('observed_status_v2', lambda x: (x == 'BROKEN').sum()),
    n_worn_strong=('observed_status_v2', lambda x: (x == 'WORN_STRONG').sum()),
    n_worn=('observed_status_v2', lambda x: (x == 'WORN').sum()),
    n_ok=('observed_status_v2', lambda x: (x == 'OK').sum()),
    n_strong_defect_comment=('comment_evidence_level', lambda x: (x == 'STRONG_DEFECT').sum()),
    n_advisory_comment=('comment_evidence_level', lambda x: (x == 'ADVISORY').sum()),
    pct_comment=('has_comment', 'mean'),
    pct_broken=('observed_status_v2', lambda x: (x == 'BROKEN').mean()),
)
non_by_checkpoint['pct_comment'] = (non_by_checkpoint['pct_comment'] * 100).round(1)
non_by_checkpoint['pct_broken']  = (non_by_checkpoint['pct_broken']  * 100).round(1)
non_by_checkpoint = non_by_checkpoint.sort_values('n_non', ascending=False).reset_index(drop=True)
print(f'Distinct checkpoints with NON: {len(non_by_checkpoint)}')
print(non_by_checkpoint.head(20).to_string(index=False))

Distinct checkpoints with NON: 6
                                checkpoint_code           tier zone_controle  n_non  n_with_comment n_critique  n_hard_cap  n_broken  n_worn_strong  n_worn  n_ok  n_strong_defect_comment  n_advisory_comment  pct_comment  pct_broken
 autres_prestations_controle_filtre_d_habitacle    T3_COSMETIC     ENTRETIEN     41              24          0           0         0              0      41     0                       13                   2        58.50        0.00
       autres_prestations_controle_filtre_a_air    T3_COSMETIC     ENTRETIEN     34              26          0           0         0              0      34     0                       18                   4        76.50        0.00
autres_prestations_fonctionnement_climatisation    T3_COSMETIC     ENTRETIEN     31              16          0           0         0              0      31     0                        5                   0        51.60        0.00
       autres_prestations_operation_d_e

### NON-4 — Contextual Subsets

`NON` rows are split into three sub-populations based on anomaly flags,
which determine what VHS status they could plausibly map to:

| Sub-population | Condition | Plausible target status |
|---|---|---|
| `df_non_simple` | `est_anomalie_critique = false` AND `est_anomalie_securite = false` | WORN |
| `df_non_critical` | `est_anomalie_critique = true` | WORN_STRONG |
| `df_non_no_anomaly` | no anomaly flag at all, `tier` not T1 | IGNORED / UNKNOWN |

In [420]:
# Sub-population 1: NON with no anomaly flags -> plausible WORN candidate
_no_crit = ~df_non['est_anomalie_critique'].fillna(False) if 'est_anomalie_critique' in df_non.columns else pd.Series(True, index=df_non.index)
_no_sec  = ~df_non['est_anomalie_securite'].fillna(False) if 'est_anomalie_securite' in df_non.columns else pd.Series(True, index=df_non.index)
df_non_simple = df_non[_no_crit & _no_sec].copy()
print(f'df_non_simple (WORN candidate): {len(df_non_simple):,} rows')
print(df_non_simple['observed_status_v2'].value_counts().to_string())

df_non_simple (WORN candidate): 149 rows
observed_status_v2
WORN    128


In [421]:
# Sub-population 2: NON + est_anomalie_critique=True → plausible WORN_STRONG candidate
df_non_critical = df_non[df_non['est_anomalie_critique'].fillna(False)].copy()
print(f'df_non_critical (WORN_STRONG candidate): {len(df_non_critical):,} rows')
print(df_non_critical['observed_status_v2'].value_counts().to_string())
print()
print('Evidence level in critical NON rows:')
print(df_non_critical['comment_evidence_level'].value_counts().to_string())

df_non_critical (WORN_STRONG candidate): 0 rows
Series([], )

Evidence level in critical NON rows:
Series([], )


In [422]:
# Sub-population 3: NON with no anomaly flag AND tier != T1 -> possibly IGNORED/UNKNOWN
_no_crit2 = ~df_non['est_anomalie_critique'].fillna(False) if 'est_anomalie_critique' in df_non.columns else pd.Series(True, index=df_non.index)
_no_sec2  = ~df_non['est_anomalie_securite'].fillna(False) if 'est_anomalie_securite' in df_non.columns else pd.Series(True, index=df_non.index)
df_non_no_anomaly = df_non[_no_crit2 & _no_sec2 & (df_non['tier'] != 'T1')].copy()
print(f'df_non_no_anomaly (UNKNOWN/IGNORED): {len(df_non_no_anomaly):,} rows')
if len(df_non_no_anomaly) > 0:
    print(df_non_no_anomaly['zone_controle'].value_counts().head(10).to_string())

df_non_no_anomaly (UNKNOWN/IGNORED): 149 rows
zone_controle
ENTRETIEN    149


### NON-5 — NON Rows with Strong Defect Comments

These rows combine a `NON` structured value with a comment that mentions concrete
defects (brake, leak, change required, missing level, suspension).
They are the strongest candidates for an upgrade toward `BROKEN` or `WORN_STRONG`.

In [423]:
_non_strong_mask = (df_non["comment_evidence_level"] == "STRONG_DEFECT")
non_with_strong_comments = df_non[_non_strong_mask].copy()
print(f"NON rows with STRONG_DEFECT comment: {len(non_with_strong_comments):,}")
if len(non_with_strong_comments) > 0:
    print()
    print("Status breakdown:")
    print(non_with_strong_comments["observed_status_v2"].value_counts().to_string())
    print()
    print("Top checkpoints:")
    _top_col = "libelle_checkpoint" if "libelle_checkpoint" in non_with_strong_comments.columns else "checkpoint_code"
    print(non_with_strong_comments[_top_col].value_counts().head(10).to_string())
    print()
    STRONG_COLS = [c for c in [
        "checkpoint_code", "libelle_checkpoint", "zone_controle",
        "commentaire_original", "detected_keywords",
        "est_anomalie_critique", "observed_status_v2", "decision_v2"
    ] if c in non_with_strong_comments.columns]
    print(non_with_strong_comments[STRONG_COLS].head(10).to_string(index=False))

NON rows with STRONG_DEFECT comment: 44

Status breakdown:
observed_status_v2
WORN    41

Top checkpoints:
checkpoint_code
autres_prestations_controle_filtre_a_air           18
autres_prestations_controle_filtre_d_habitacle     13
autres_prestations_fonctionnement_climatisation     5
autres_prestations_courroie_de_disribution          4
autres_prestations_operation_d_entretien            3
autres_prestations_controle_bougies_d_allumage      1

                                checkpoint_code zone_controle                                                                               commentaire_original              detected_keywords est_anomalie_critique observed_status_v2 decision_v2
       autres_prestations_controle_filtre_a_air     ENTRETIEN                                                Filtre a air a remplacer les Bougies sont remplacer                change_required                 False               WORN          OK
       autres_prestations_controle_filtre_a_air     ENTRETIEN 

### NON-6 — NON Rows with Advisory or Unclear Comments

`NON` + `ADVISORY` comment: the inspector suggested action but did not confirm a defect.
`NON` + `WEAK_OR_UNCLEAR` comment: the comment exists but provides no clear signal.
`NON` + `NO_COMMENT`: the most common case — structured value alone must carry the meaning.

In [424]:
_non_adv_mask = df_non["comment_evidence_level"].isin(["ADVISORY", "WEAK_OR_UNCLEAR"])
_non_nocomm   = df_non["comment_evidence_level"] == "NO_COMMENT"

non_with_advisory_or_unclear_comments = df_non[_non_adv_mask].copy()
print(f'NON + ADVISORY or WEAK_OR_UNCLEAR comment: {len(non_with_advisory_or_unclear_comments):,}')
print(non_with_advisory_or_unclear_comments['comment_evidence_level'].value_counts().to_string())
print()
print(f'NON + NO_COMMENT: {int(_non_nocomm.sum()):,}')
print()
if len(non_with_advisory_or_unclear_comments) > 0:
    ADV_COLS = ['checkpoint_code', 'libelle_checkpoint',
                'commentaire_original', 'detected_keywords',
                'comment_evidence_level', 'observed_status_v2', 'decision_v2']
    ADV_COLS = [c for c in ADV_COLS if c in non_with_advisory_or_unclear_comments.columns]
    print(non_with_advisory_or_unclear_comments[ADV_COLS].head(8).to_string(index=False))

NON + ADVISORY or WEAK_OR_UNCLEAR comment: 41
comment_evidence_level
WEAK_OR_UNCLEAR    34
ADVISORY            7

NON + NO_COMMENT: 64

                                checkpoint_code                                    commentaire_original       detected_keywords comment_evidence_level observed_status_v2 decision_v2
       autres_prestations_controle_filtre_a_air                                       Vidange conseillé action_repair, advisory               ADVISORY               WORN     DEGRADE
       autres_prestations_controle_filtre_a_air           Climatiseur ne fonctionne pas Vidange a faire           action_repair        WEAK_OR_UNCLEAR               WORN     DEGRADE
 autres_prestations_controle_filtre_d_habitacle           Climatiseur ne fonctionne pas Vidange a faire           action_repair        WEAK_OR_UNCLEAR               WORN     DEGRADE
autres_prestations_fonctionnement_climatisation           Climatiseur ne fonctionne pas Vidange a faire           action_repair        W

### NON-7 — NON in Severe Inspection Cases

These are `NON` rows where the final vehicle decision is `CRITIQUE` or `IMMOBILISE`.
The question is whether the `NON` observation materially contributed to that outcome
(direct hard-cap) or was incidental (low-penalty row in a severe inspection).

In [425]:
_non_severe = df_non["decision_v2"].isin(["CRITIQUE", "IMMOBILISE"])
non_in_severe_cases = df_non[_non_severe].copy()
print(f'NON rows in CRITIQUE/IMMOBILISE inspections: {len(non_in_severe_cases):,}')
print()
print('decision_v2 breakdown:')
print(non_in_severe_cases['decision_v2'].value_counts().to_string())
print()
print('observed_status_v2 breakdown:')
print(non_in_severe_cases['observed_status_v2'].value_counts().to_string())
print()
print('comment_evidence_level breakdown:')
print(non_in_severe_cases['comment_evidence_level'].value_counts().to_string())
print()
print('is_hard_cap_trigger in severe NON rows:')
print(non_in_severe_cases['is_hard_cap_trigger'].value_counts().to_string())

NON rows in CRITIQUE/IMMOBILISE inspections: 29

decision_v2 breakdown:
decision_v2
CRITIQUE      27
IMMOBILISE     2

observed_status_v2 breakdown:
observed_status_v2
WORN    26

comment_evidence_level breakdown:
comment_evidence_level
NO_COMMENT         19
STRONG_DEFECT       5
WEAK_OR_UNCLEAR     3
ADVISORY            2

is_hard_cap_trigger in severe NON rows:
is_hard_cap_trigger
False    29


#### Interpretation Guide

| Observation | Interpretation |
|---|---|
| Most severe-case `NON` rows are `is_hard_cap_trigger = False` | `NON` is incidental; it rides on other BROKEN rows |
| Most severe-case `NON` rows are `is_hard_cap_trigger = True` | `NON` itself triggers hard caps → current `BROKEN` mapping is appropriate |
| Severe-case `NON` rows carry `STRONG_DEFECT` comments | Comment confirms criticality |
| Severe-case `NON` rows carry `NO_COMMENT` or `ADVISORY` | Severity comes from structured fields, not comment |

### NON-8 — NON Hard-Cap Triggers

Direct hard-cap triggers: `NON` rows where `is_hard_cap_trigger = True`.
These rows are definitively high-severity and should remain `BROKEN`.

In [426]:
_hc_col = "is_hard_cap_trigger"
if _hc_col not in df_non.columns:
    print("is_hard_cap_trigger column not available — skipping.")
else:
    non_hard_cap_triggers = df_non[df_non[_hc_col].fillna(False)].copy()
    print(f"NON rows that are hard-cap triggers: {len(non_hard_cap_triggers):,}")
    if len(non_hard_cap_triggers) > 0:
        HC_COLS = [c for c in [
            "checkpoint_code", "libelle_checkpoint", "zone_controle", "tier",
            "commentaire_original", "detected_keywords",
            "est_anomalie_critique", "observed_status_v2", "decision_v2", "penalty_v2"
        ] if c in non_hard_cap_triggers.columns]
        print(non_hard_cap_triggers[HC_COLS].to_string(index=False))
    else:
        print("None — NON is not a direct hard-cap trigger in this run.")
    non_hard_cap_triggers  # expose variable for export cell

NON rows that are hard-cap triggers: 0
None — NON is not a direct hard-cap trigger in this run.


### NON-9 — Simulation: What If NON → WORN_STRONG?

A pandas-only simulation (no DB writes, no VHS recomputation).

**Hypothesis:** treat all `NON` rows currently mapped to `BROKEN` as `WORN_STRONG` instead.
The simulated penalty is `(penalty_worn + penalty_broken) / 2`, matching the V2 formula.

We compute `penalty_delta` = simulated − current and summarise the impact.

> This simulation is **illustrative only**. It does not produce a new run ID.
> It does not account for inter-row dependencies (e.g. a reduced penalty may no longer
> trigger a hard cap, changing the vehicle-level decision in ways not captured here).

In [427]:
# ── Robust column resolution ─────────────────────────────────────────────
# penalty column
if 'current_penalty_v2' in df_non.columns:
    _pen_col = 'current_penalty_v2'
elif 'penalty_applied_v2' in df_non.columns:
    df_non = df_non.copy()
    df_non['current_penalty_v2'] = df_non['penalty_applied_v2']
    _pen_col = 'current_penalty_v2'
elif 'penalty_v2' in df_non.columns:
    df_non = df_non.copy()
    df_non['current_penalty_v2'] = df_non['penalty_v2']
    _pen_col = 'current_penalty_v2'
else:
    df_non = df_non.copy()
    df_non['current_penalty_v2'] = 0
    _pen_col = 'current_penalty_v2'
    print('Warning: no V2 penalty column found. current_penalty_v2 set to 0 for simulation.')

# penalty_worn / penalty_broken
if 'penalty_worn' not in df_non.columns:
    df_non['penalty_worn'] = 0
    print('Warning: penalty_worn missing; defaulted to 0.')
if 'penalty_broken' not in df_non.columns:
    df_non['penalty_broken'] = 0
    print('Warning: penalty_broken missing; defaulted to 0.')

# observed_status_v2
if 'observed_status_v2' not in df_non.columns:
    df_non['observed_status_v2'] = 'UNKNOWN'
    print('Warning: observed_status_v2 missing; defaulted to UNKNOWN.')

# hard_cap_type_v2
if 'hard_cap_type_v2' not in df_non.columns:
    df_non['hard_cap_type_v2'] = None

# is_hard_cap_trigger
if 'is_hard_cap_trigger' not in df_non.columns:
    df_non['is_hard_cap_trigger'] = False
    print('Warning: is_hard_cap_trigger missing; defaulted to False.')

# est_anomalie / est_anomalie_critique
if 'est_anomalie' not in df_non.columns:
    if 'est_anomalie_critique' in df_non.columns:
        df_non['est_anomalie'] = df_non['est_anomalie_critique']
    else:
        df_non['est_anomalie'] = False
if 'est_anomalie_critique' not in df_non.columns:
    df_non['est_anomalie_critique'] = False

# checkpoint_libelle
if 'checkpoint_libelle' not in df_non.columns:
    if 'libelle_checkpoint' in df_non.columns:
        df_non['checkpoint_libelle'] = df_non['libelle_checkpoint']
    else:
        df_non['checkpoint_libelle'] = df_non['checkpoint_code']

# immatriculation_norm
if 'immatriculation_norm' not in df_non.columns:
    df_non['immatriculation_norm'] = None

# ── Build simulation dataframe ────────────────────────────────────────────
SIM_COLS = [
    'inspection_key', 'immatriculation_norm', 'checkpoint_code', 'checkpoint_libelle',
    'zone_controle', 'tier', 'valeur_controle',
    'est_anomalie', 'est_anomalie_critique',
    'observed_status_v2',
    'current_penalty_v2', 'penalty_worn', 'penalty_broken',
    'safety_grade_v2', 'decision_v2', 'hard_cap_type_v2', 'is_hard_cap_trigger',
    'commentaire_original', 'comment_evidence_level', 'detected_keywords',
]
SIM_COLS = [c for c in SIM_COLS if c in df_non.columns]
df_sim = df_non[SIM_COLS].copy()
df_sim = df_sim.rename(columns={'observed_status_v2': 'current_observed_status_v2'})

# ── Apply simulation rules ────────────────────────────────────────────────
# Rule 1: BROKEN -> WORN_STRONG, penalty = (worn + broken) / 2
# Rule 2: WORN_STRONG -> stays WORN_STRONG, penalty unchanged
# Rule 3: WORN -> stays WORN, penalty unchanged
# Rule 4: anything else -> unchanged
_mask_broken = df_sim['current_observed_status_v2'] == 'BROKEN'

df_sim['simulated_status']  = df_sim['current_observed_status_v2'].copy()
df_sim['simulated_penalty'] = df_sim['current_penalty_v2'].copy()

df_sim.loc[_mask_broken, 'simulated_status'] = 'WORN_STRONG'
df_sim.loc[_mask_broken, 'simulated_penalty'] = (
    (df_sim.loc[_mask_broken, 'penalty_worn'] + df_sim.loc[_mask_broken, 'penalty_broken']) / 2
).fillna(0)

df_sim['penalty_delta'] = df_sim['current_penalty_v2'] - df_sim['simulated_penalty']

# Reorder columns to match spec
FINAL_SIM_COLS = [
    'inspection_key', 'immatriculation_norm', 'checkpoint_code', 'checkpoint_libelle',
    'zone_controle', 'tier', 'valeur_controle',
    'est_anomalie', 'est_anomalie_critique',
    'current_observed_status_v2', 'simulated_status',
    'current_penalty_v2', 'simulated_penalty', 'penalty_delta',
    'safety_grade_v2', 'decision_v2', 'hard_cap_type_v2', 'is_hard_cap_trigger',
    'commentaire_original', 'comment_evidence_level', 'detected_keywords',
]
FINAL_SIM_COLS = [c for c in FINAL_SIM_COLS if c in df_sim.columns]
non_simulation_if_not_broken = df_sim[FINAL_SIM_COLS].reset_index(drop=True)

print('Simulation complete. Shape:', non_simulation_if_not_broken.shape)
print(non_simulation_if_not_broken[['current_observed_status_v2','simulated_status',
    'current_penalty_v2','simulated_penalty','penalty_delta']].head(10).to_string(index=False))

Simulation complete. Shape: (149, 21)
current_observed_status_v2 simulated_status  current_penalty_v2  simulated_penalty  penalty_delta
                      WORN             WORN                   0                  0              0
                      WORN             WORN                   0                  0              0
                      WORN             WORN                   0                  0              0
                      WORN             WORN                   0                  0              0
                      WORN             WORN                   0                  0              0
                      WORN             WORN                   0                  0              0
                      WORN             WORN                   0                  0              0
                      WORN             WORN                   0                  0              0
                      WORN             WORN                   0                 

In [428]:
# ── Simulation summary ───────────────────────────────────────────────────
sim = non_simulation_if_not_broken

_n_non_total    = len(sim)
_n_broken       = int((sim['current_observed_status_v2'] == 'BROKEN').sum())
_n_worn         = int((sim['current_observed_status_v2'] == 'WORN').sum())
_n_worn_strong  = int((sim['current_observed_status_v2'] == 'WORN_STRONG').sum())
_tot_cur_pen    = float(sim['current_penalty_v2'].sum())
_tot_sim_pen    = float(sim['simulated_penalty'].sum())
_tot_reduction  = float(sim['penalty_delta'].sum())
_n_critique     = int((sim['decision_v2'] == 'CRITIQUE').sum()) if 'decision_v2' in sim.columns else 0
_n_immobilise   = int((sim['decision_v2'] == 'IMMOBILISE').sum()) if 'decision_v2' in sim.columns else 0
_n_hardcap      = int(sim['is_hard_cap_trigger'].sum()) if 'is_hard_cap_trigger' in sim.columns else 0

print('=== NON Simulation Summary ===')
print(f'Total NON rows                : {_n_non_total:>6,}')
print(f'  currently BROKEN            : {_n_broken:>6,}')
print(f'  currently WORN              : {_n_worn:>6,}')
print(f'  currently WORN_STRONG       : {_n_worn_strong:>6,}')
print(f'Total current penalty (NON)   : {_tot_cur_pen:>10.2f}')
print(f'Total simulated penalty (NON) : {_tot_sim_pen:>10.2f}')
print(f'Total penalty reduction       : {_tot_reduction:>10.2f}')
print(f'NON rows in CRITIQUE          : {_n_critique:>6,}')
print(f'NON rows in IMMOBILISE        : {_n_immobilise:>6,}')
print(f'NON hard-cap trigger rows     : {_n_hardcap:>6,}')

=== NON Simulation Summary ===
Total NON rows                :    149
  currently BROKEN            :      0
  currently WORN              :    128
  currently WORN_STRONG       :      0
Total current penalty (NON)   :       0.00
Total simulated penalty (NON) :       0.00
Total penalty reduction       :       0.00
NON rows in CRITIQUE          :     27
NON rows in IMMOBILISE        :      2
NON hard-cap trigger rows     :      0


## Final interpretation of NON

Based on the current dataset, **NON should not be treated as BROKEN by default**.

---

### Observed facts (current run)

- NON appears in **149 rows**.
- NON appears in **73 inspections**.
- All NON rows are located in the **ENTRETIEN zone**.
- NON does **not** appear in T1_VITAL or T1_IMPORTANT checkpoints.
- NON has **0 rows** where `est_anomalie_critique = true`.
- NON has **0 hard-cap trigger rows**.
- NON may appear in CRITIQUE or IMMOBILISE inspections, but it is **not the direct
  cause** of those decisions because `is_hard_cap_trigger` is always false.

---

### Business interpretation

`NON` is a **negative answer** or a **non-conformity indicator**, especially in the
maintenance/service section (`zone_controle = ENTRETIEN`). It does not necessarily
mean that a component is broken.

Therefore, **NON should not automatically be mapped to BROKEN**.

---

### Recommended interpretation (current data)

| Condition | Recommended status |
|---|---|
| NON + `est_anomalie = true` + `est_anomalie_critique = false` | **WORN** |
| NON + `est_anomalie = false` + `est_anomalie_critique = false` | **UNKNOWN** (not scored; depends on checkpoint context) |

---

### Future-safe rule (preventive, not observed)

**NON + `est_anomalie_critique = true` → WORN_STRONG**

Although **no case of NON + est_anomalie_critique=true was observed in the current dataset**,
this combination is included in the target interpretation table to cover possible future data.
It should be interpreted as **WORN_STRONG** rather than BROKEN, because NON expresses a
negative answer or non-conformity, but does not explicitly confirm that a component is broken.

---

### Options for VHS V3

#### Option A — Remap NON by context *(recommended based on current data)*

| Condition | Target status |
|---|---|
| NON + `est_anomalie_critique = true` | WORN_STRONG *(preventive)* |
| NON + `est_anomalie = true` + `est_anomalie_critique = false` | WORN |
| NON + `est_anomalie = false` (non-critical service check) | UNKNOWN / IGNORED |

**Prerequisite:** domain expert validation per checkpoint_code before implementing.

#### Option B — Keep NON → BROKEN *(conservative)*

Apply only if future data shows that NON is associated with hard-cap triggers or
`est_anomalie_critique = true` at a significant rate.

#### Option C — Defer and flag for manual review

Keep BROKEN for now; add a `needs_review` flag in `dim_checkpoint` for NON checkpoints
and revisit after the next inspector training cycle.

### NON — Export: CSV Files

Ten CSV files written to `data/quality_reports/vhs/staffim_comment_analysis/`.
No database table is touched.

In [429]:
import csv as _csv
from pathlib import Path as _Path

_OUT = _Path(r"data/quality_reports/vhs/staffim_comment_analysis")
_OUT.mkdir(parents=True, exist_ok=True)

def _save(df, filename):
    path = _OUT / filename
    df.to_csv(path, index=False, encoding="utf-8-sig", quoting=_csv.QUOTE_NONNUMERIC)
    print(f"  Saved {len(df):>6,} rows  ->  {filename}")

# Ensure variables exist (fallback to empty df if a cell was skipped)
_fallback = df_non.iloc[0:0].copy()
if "non_hard_cap_triggers"              not in dir(): non_hard_cap_triggers              = _fallback
if "non_with_strong_comments"           not in dir(): non_with_strong_comments           = _fallback
if "non_with_advisory_or_unclear_comments" not in dir(): non_with_advisory_or_unclear_comments = _fallback
if "non_in_severe_cases"                not in dir(): non_in_severe_cases                = _fallback
if "non_simulation_if_not_broken"       not in dir(): non_simulation_if_not_broken       = _fallback
if "df_non_simple"                      not in dir(): df_non_simple                      = _fallback
if "df_non_critical"                    not in dir(): df_non_critical                    = _fallback
if "df_non_no_anomaly"                  not in dir(): df_non_no_anomaly                  = _fallback

# ── Profile summary dataframe ────────────────────────────────────────────
_n_non  = len(df_non)
_n_ins  = df_non["inspection_key"].nunique()
_n_veh  = df_non["immatriculation_norm"].nunique() if "immatriculation_norm" in df_non.columns else 0
_n_cmt  = int(df_non["has_comment"].sum()) if "has_comment" in df_non.columns else 0
_n_crit = int(df_non["est_anomalie_critique"].sum()) if "est_anomalie_critique" in df_non.columns else 0
_n_hc   = int(df_non["is_hard_cap_trigger"].sum()) if "is_hard_cap_trigger" in df_non.columns else 0
_n_cri_dec  = int((df_non["decision_v2"] == "CRITIQUE").sum())   if "decision_v2" in df_non.columns else 0
_n_imm_dec  = int((df_non["decision_v2"] == "IMMOBILISE").sum()) if "decision_v2" in df_non.columns else 0
_tot_cur    = float(non_simulation_if_not_broken["current_penalty_v2"].sum())  if "current_penalty_v2" in non_simulation_if_not_broken.columns else 0
_tot_sim    = float(non_simulation_if_not_broken["simulated_penalty"].sum())   if "simulated_penalty"  in non_simulation_if_not_broken.columns else 0
_tot_red    = float(non_simulation_if_not_broken["penalty_delta"].sum())       if "penalty_delta"      in non_simulation_if_not_broken.columns else 0

import pandas as _pd
non_profile_summary = _pd.DataFrame([
    {"metric": "total_non_rows",               "value": _n_non},
    {"metric": "distinct_inspections",         "value": _n_ins},
    {"metric": "distinct_vehicles",            "value": _n_veh},
    {"metric": "rows_with_comment",            "value": _n_cmt},
    {"metric": "rows_without_comment",         "value": _n_non - _n_cmt},
    {"metric": "est_anomalie_critique_true",   "value": _n_crit},
    {"metric": "hard_cap_trigger_rows",        "value": _n_hc},
    {"metric": "rows_in_critique_decision",    "value": _n_cri_dec},
    {"metric": "rows_in_immobilise_decision",  "value": _n_imm_dec},
    {"metric": "non_broken_rows",              "value": int((non_simulation_if_not_broken["current_observed_status_v2"] == "BROKEN").sum()) if "current_observed_status_v2" in non_simulation_if_not_broken.columns else 0},
    {"metric": "non_worn_rows",                "value": int((non_simulation_if_not_broken["current_observed_status_v2"] == "WORN").sum())   if "current_observed_status_v2" in non_simulation_if_not_broken.columns else 0},
    {"metric": "total_current_penalty",        "value": round(_tot_cur, 2)},
    {"metric": "total_simulated_penalty",      "value": round(_tot_sim, 2)},
    {"metric": "total_penalty_reduction",      "value": round(_tot_red, 2)},
    {"metric": "recommended_status_simple",    "value": "WORN"},
    {"metric": "recommended_status_no_anomaly","value": "UNKNOWN"},
    {"metric": "recommended_status_critical",  "value": "WORN_STRONG (preventive)"},
])

# ── CSV exports ──────────────────────────────────────────────────────────
print("=== NON audit CSV exports ===")
_save(non_profile_summary,                   "non_profile_summary.csv")
_save(non_by_checkpoint,                     "non_by_checkpoint.csv")
_save(df_non_simple,                         "non_simple_cases.csv")
_save(df_non_critical,                       "non_critical_cases.csv")
_save(df_non_no_anomaly,                     "non_no_anomaly_cases.csv")
_save(non_with_strong_comments,              "non_with_strong_comments.csv")
_save(non_with_advisory_or_unclear_comments, "non_with_advisory_or_unclear_comments.csv")
_save(non_in_severe_cases,                   "non_in_severe_cases.csv")
_save(non_hard_cap_triggers,                 "non_hard_cap_triggers.csv")
_save(non_simulation_if_not_broken,          "non_simulation_if_not_broken.csv")

# ── Update staffim_comment_analysis_summary.md ───────────────────────────
_md_path = _OUT / "staffim_comment_analysis_summary.md"
_non_section = "\n".join([
    "",
    "---",
    "",
    "## Audit of NON value",
    "",
    f"| Metric | Value |",
    f"|--------|-------|",
    f"| Total NON rows | {_n_non:,} |",
    f"| Distinct inspections | {_n_ins:,} |",
    f"| Distinct vehicles | {_n_veh:,} |",
    f"| Rows with comment | {_n_cmt:,} |",
    f"| Rows without comment | {_n_non - _n_cmt:,} |",
    "",
    "**Distribution by zone:** all NON rows are in zone `ENTRETIEN`.",
    "",
    "**Distribution by tier:** NON appears in T3_COSMETIC, UNKNOWN_REVIEW, T2_CRITICAL, T2_NORMAL.",
    "NON does not appear in T1_VITAL or T1_IMPORTANT.",
    "",
    "**Distribution by anomaly flags:**",
    f"- est_anomalie_critique = true: {_n_crit}",
    f"- hard-cap trigger rows: {_n_hc}",
    "",
    f"**NON rows in CRITIQUE decisions:** {_n_cri_dec}",
    f"**NON rows in IMMOBILISE decisions:** {_n_imm_dec}",
    f"**NON hard-cap triggers:** {_n_hc}",
    "",
    "**Simulation result** (NON BROKEN -> WORN_STRONG):",
    f"- Total current penalty from NON: {_tot_cur:.2f}",
    f"- Total simulated penalty from NON: {_tot_sim:.2f}",
    f"- Total penalty reduction: {_tot_red:.2f}",
    "",
    "**Final recommendation:**",
    "",
    "Based on the current data, NON should not be mapped to BROKEN by default.",
    "In the current dataset, NON is best interpreted as WORN when it is associated",
    "with a simple anomaly. If future data contains NON + est_anomalie_critique=true,",
    "it should be interpreted as WORN_STRONG and reviewed, not automatically treated as BROKEN.",
])

if _md_path.exists():
    _existing = _md_path.read_text(encoding="utf-8")
    if "## Audit of NON value" in _existing:
        # Replace existing NON section
        _start = _existing.find("\n---\n\n## Audit of NON value")
        if _start == -1:
            _start = _existing.find("## Audit of NON value")
        _end = _existing.find("\n---\n", _start + 5)
        if _end == -1:
            _end = len(_existing)
        _updated = _existing[:_start] + _non_section + _existing[_end:]
    else:
        _updated = _existing + _non_section
    _md_path.write_text(_updated, encoding="utf-8")
else:
    _md_path.write_text(_non_section, encoding="utf-8")
print(f"  Updated: {_md_path.name}")

# ── Final console output ─────────────────────────────────────────────────
print()
print("NON audit fixed successfully.")
print()
_n_broken_sim = int((non_simulation_if_not_broken["current_observed_status_v2"] == "BROKEN").sum()) if "current_observed_status_v2" in non_simulation_if_not_broken.columns else 0
_n_worn_sim   = int((non_simulation_if_not_broken["current_observed_status_v2"] == "WORN").sum())   if "current_observed_status_v2" in non_simulation_if_not_broken.columns else 0
_n_wos_sim    = int((non_simulation_if_not_broken["current_observed_status_v2"] == "WORN_STRONG").sum()) if "current_observed_status_v2" in non_simulation_if_not_broken.columns else 0
print(f"  total NON rows                   : {_n_non:,}")
print(f"  distinct inspections             : {_n_ins:,}")
print(f"  NON rows currently BROKEN        : {_n_broken_sim:,}")
print(f"  NON rows currently WORN          : {_n_worn_sim:,}")
print(f"  NON rows currently WORN_STRONG   : {_n_wos_sim:,}")
print(f"  NON rows with est_anomalie_critique=true : {_n_crit:,}")
print(f"  NON rows in CRITIQUE decisions   : {_n_cri_dec:,}")
print(f"  NON rows in IMMOBILISE decisions : {_n_imm_dec:,}")
print(f"  NON hard-cap trigger rows        : {_n_hc:,}")
print(f"  total simulated penalty reduction: {_tot_red:.2f}")
print()
print("  Final recommendation:")
print("  Based on the current data, NON should not be mapped to BROKEN by default.")
print("  NON is best interpreted as WORN when associated with a simple anomaly.")
print("  If future data contains NON + est_anomalie_critique=true, use WORN_STRONG.")

=== NON audit CSV exports ===
  Saved     17 rows  ->  non_profile_summary.csv
  Saved      6 rows  ->  non_by_checkpoint.csv
  Saved    149 rows  ->  non_simple_cases.csv
  Saved      0 rows  ->  non_critical_cases.csv
  Saved    149 rows  ->  non_no_anomaly_cases.csv
  Saved     44 rows  ->  non_with_strong_comments.csv
  Saved     41 rows  ->  non_with_advisory_or_unclear_comments.csv
  Saved     29 rows  ->  non_in_severe_cases.csv
  Saved      0 rows  ->  non_hard_cap_triggers.csv
  Saved    149 rows  ->  non_simulation_if_not_broken.csv
  Updated: staffim_comment_analysis_summary.md

NON audit fixed successfully.

  total NON rows                   : 149
  distinct inspections             : 73
  NON rows currently BROKEN        : 0
  NON rows currently WORN          : 128
  NON rows currently WORN_STRONG   : 0
  NON rows with est_anomalie_critique=true : 0
  NON rows in CRITIQUE decisions   : 27
  NON rows in IMMOBILISE decisions : 2
  NON hard-cap trigger rows        : 0
  total

---
## 10. Future Integration Strategy

### Current State of the Official VHS

The current `VHS_BALANCED_V2` score:

- Uses **structured STAFFIM values** (`valeur_controle`, `est_anomalie`, `est_anomalie_critique`) as its only inputs
- Enriches those inputs with **checkpoint criticality metadata** from `mart.dim_checkpoint` (tier, penalty weights, boolean flags)
- Is fully **rule-based, explainable, and auditable** — every penalty can be traced to a specific checkpoint observation
- **Does not use `commentaire_zone`** in any form

### Why Comments Cannot Be Integrated Directly

| Risk | Detail |
|------|--------|
| **Inconsistency** | Same defect described differently by different inspectors |
| **Abbreviations and spelling** | `essui` / `essuie`, `durits` / `durites`, `patin` / `plaquette` — keyword recall is imperfect |
| **Ambiguous scope** | `"patin av / amortisseur av / fuite huile moteur"` spans three different checkpoints in one comment |
| **Repaired items** | `"essuie-glace change"` = completed repair, not a current defect — incorrectly scored as defect if taken at face value |
| **Advisory inflation** | `"vidange conseille"` is a suggestion; direct integration would inflate severity for advisory observations |
| **Explainability** | Free-text inputs reduce auditability and complicate regulatory justification for fraud decisions |

### Recommended Integration Roadmap

```
Phase 1 (current — V2)
  Rule-based VHS from structured fields only.
  Comments stored in the data model but never scored.

Phase 2 (next — audit layer)
  comment_evidence_level stored in a side table (not in fact_vhs_score).
  Surfaced in the inspection review UI for analyst review.
  Trigger: flag borderline CRITIQUE cases where evidence = ADVISORY.

Phase 3 (later — annotation)
  Human annotators review comment + structured value pairs.
  Labels: CONFIRMS_DEFECT | CHALLENGES_SEVERITY | UNRELATED | REPAIR_DONE.
  Target: 500-1000 annotated pairs to build a validated training set.

Phase 4 (future — supervised signal)
  Rule-engine or lightweight classifier trained on Phase 3 annotations.
  Output stored as evidence label + confidence in mart.fact_comment_evidence.
  Never overwrites structured VHS values.
  Human review required before any score adjustment.
```

### Key Governance Principle

> Any signal derived from free-text **must be stored as evidence**, not as a direct scoring input. The official VHS must remain rule-based and fully auditable for regulatory compliance. Comment evidence is a support layer, not a scoring layer.

---
## 11. Optional Future Agent Design — Not Used in Current Score

> **This section is conceptual only.** No agent is implemented in this notebook. No agent output is stored anywhere. The official VHS is completely unaffected.

### Motivation

Once a sufficient annotated comment dataset is available (Phase 3 above), a small language model agent could process new comments at scale and return a structured evidence object for human review.

### What the Agent Would Do

1. Read `commentaire_zone` for a specific checkpoint observation
2. Identify the most likely related vehicle component
3. Classify the comment as STRONG_DEFECT, ADVISORY, REPAIR_DONE, or UNCLEAR
4. Return a structured JSON evidence object — **never a score**

### Proposed Agent Output Schema

```json
{
  "inspection_key": "INS-2025-XXXXX",
  "checkpoint_code": "CP_FREINS_AV",
  "comment_excerpt": "patin av a changer / amortisseur av fuite",
  "suggested_evidence_level": "STRONG_DEFECT",
  "suggested_related_component": "freinage / suspension avant",
  "confidence": 0.87,
  "explanation": "Comment explicitly mentions brake pad replacement required
    and a front shock absorber leak — both are safety-critical components."
}
```

### Design Constraints

| Constraint | Rationale |
|------------|-----------|
| Agent output stored in `mart.fact_comment_evidence` (proposed), never in `fact_vhs_score` | Preserve official score integrity |
| Agent confidence must exceed threshold before flagging (e.g. > 0.75) | Reduce false positives |
| Human reviewer must approve before any score adjustment | Regulatory auditability |
| Agent retrained only on validated annotation data | Prevent drift from unannotated noise |
| Agent version tracked alongside VHS `run_id` | Reproducibility |

### Recommended Model

- **Development / low-volume:** `claude-haiku-4-5-20251001` — fast, cost-efficient, sufficient for short comment classification
- **Production / high-accuracy:** `claude-sonnet-4-6` — stronger reasoning for ambiguous multi-component comments
- Output format: structured JSON via tool_use to guarantee parseable responses
- Storage: post-inspection batch job; results written to `mart.fact_comment_evidence`

### Proposed `mart.fact_comment_evidence` Schema

```sql
CREATE TABLE mart.fact_comment_evidence (
    inspection_key          TEXT NOT NULL,
    checkpoint_code         TEXT NOT NULL,
    agent_run_id            TEXT NOT NULL,
    agent_version           TEXT NOT NULL,
    comment_excerpt         TEXT,
    suggested_evidence_level TEXT,   -- STRONG_DEFECT | ADVISORY | REPAIR_DONE | UNCLEAR
    suggested_component     TEXT,
    confidence              NUMERIC(4,3),
    explanation             TEXT,
    reviewed_by_human       BOOLEAN DEFAULT FALSE,
    human_label             TEXT,
    created_at              TIMESTAMP,
    PRIMARY KEY (inspection_key, checkpoint_code, agent_run_id)
);
```

> **This table does not exist yet.** It is proposed here as a design artefact for the PFE report. Creating it requires a separate DWH migration ticket.

---
## 12. Export Reports

All audit dataframes and cross-tab results are saved as CSV files under:

```
data/quality_reports/vhs/staffim_comment_analysis/
```

A Markdown summary report is also generated in the same directory.

> **Read-only reminder:** only local files are written. No database table is touched.

In [430]:
def save_csv(df: pd.DataFrame, filename: str) -> None:
    """Save dataframe to OUTPUT_DIR as UTF-8 CSV with BOM for Excel compatibility."""
    path = OUTPUT_DIR / filename
    df.to_csv(path, index=False, encoding='utf-8-sig', quoting=csv.QUOTE_NONNUMERIC)
    print(f'  Saved {len(df):>6,} rows  ->  {filename}')

print(f'Output directory: {OUTPUT_DIR}')
print(f'Writing CSV exports...')

Output directory: C:\Users\wiem\Downloads\Projet PFE\IRIS_AUTO_FRAUD\data\quality_reports\vhs\staffim_comment_analysis
Writing CSV exports...


In [431]:
# 1. Global profile summary
profile_rows = [
    {'metric': 'run_id',                   'value': RUN_ID},
    {'metric': 'total_checkpoint_rows',     'value': len(df_staffim)},
    {'metric': 'distinct_inspections',      'value': df_staffim['inspection_key'].nunique()},
    {'metric': 'distinct_vehicles',         'value': df_staffim['immatriculation_norm'].nunique()},
    {'metric': 'rows_with_comment',         'value': int(df_staffim['has_comment'].sum())},
    {'metric': 'pct_with_comment',          'value': round(100 * df_staffim['has_comment'].mean(), 2)},
]
for col in KEYWORDS:
    profile_rows.append({'metric': col + '_count', 'value': int(df_staffim[col].sum())})

ev_dist = df_staffim['comment_evidence_level'].value_counts()
for level, cnt in ev_dist.items():
    profile_rows.append({'metric': 'evidence_' + level, 'value': int(cnt)})

staffim_comment_profile = pd.DataFrame(profile_rows)
save_csv(staffim_comment_profile, 'staffim_comment_profile.csv')

  Saved     20 rows  ->  staffim_comment_profile.csv


In [432]:
# 2. Audit dataframes
save_csv(comments_with_ok_value,                  'comments_with_ok_value.csv')
save_csv(proposition_faite_comment_context,       'proposition_faite_comment_context.csv')
save_csv(intervention_conseillee_comment_context, 'intervention_conseillee_comment_context.csv')
save_csv(critique_with_advisory_comments,         'critique_with_advisory_comments.csv')
save_csv(critical_comment_evidence,               'critical_comment_evidence.csv')

# 3. Examples
save_csv(staffim_comment_examples, 'staffim_comment_examples.csv')

  Saved  2,119 rows  ->  comments_with_ok_value.csv
  Saved    105 rows  ->  proposition_faite_comment_context.csv
  Saved    575 rows  ->  intervention_conseillee_comment_context.csv
  Saved      6 rows  ->  critique_with_advisory_comments.csv
  Saved    454 rows  ->  critical_comment_evidence.csv
  Saved     22 rows  ->  staffim_comment_examples.csv


In [433]:
# 4. Cross-tab results consolidated into one CSV
crosstab_parts = []
for dim_name, ct_df in [
    ('valeur_controle',    ct_valeur),
    ('observed_status_v2', ct_status),
    ('safety_grade_v2',    ct_grade),
    ('decision_v2',        ct_decision),
    ('tier',               ct_tier),
]:
    tmp = ct_df.reset_index().copy()
    tmp.insert(0, 'dimension', dim_name)
    crosstab_parts.append(tmp)

staffim_comment_crosstabs = pd.concat(crosstab_parts, ignore_index=True)
save_csv(staffim_comment_crosstabs, 'staffim_comment_crosstabs.csv')

print('\nAll CSV exports complete.')

  Saved     29 rows  ->  staffim_comment_crosstabs.csv

All CSV exports complete.


In [434]:
# 5. Markdown summary report
n_total    = len(df_staffim)
n_commented = int(df_staffim['has_comment'].sum())
pct_cmt    = round(100 * n_commented / n_total, 1)

ev_counts = df_staffim['comment_evidence_level'].value_counts()

def fmt(n): return f'{n:,}'

md_lines = [
    '# STAFFIM Comment Analysis — Summary Report',
    '',
    f'**Run ID:** `{RUN_ID}`  ',
    f'**Profile:** `{PROFILE_NAME}`  ',
    '**Analysis date:** 2026-07-02  ',
    '',
    '---',
    '',
    '## Dataset',
    '',
    '| Metric | Value |',
    '|--------|-------|',
    f'| Total checkpoint rows | {fmt(n_total)} |',
    f'| Distinct inspections | {fmt(df_staffim["inspection_key"].nunique())} |',
    f'| Distinct vehicles | {fmt(df_staffim["immatriculation_norm"].nunique())} |',
    f'| Rows with non-empty comment | {fmt(n_commented)} ({pct_cmt}%) |',
    '',
    '## Audit Findings',
    '',
    '| Audit | Description | Rows |',
    '|-------|-------------|------|',
    f'| A | OK/Bon value with defect keyword in comment | {fmt(len(comments_with_ok_value))} |',
    f'| B | PROPOSITION FAITE (all) | {fmt(len(proposition_faite_comment_context))} |',
    f'| B-advisory | PROPOSITION FAITE — advisory comment only | {fmt(n_prop_adv)} |',
    f'| B-strong | PROPOSITION FAITE — strong defect in comment | {fmt(n_prop_strong)} |',
    f'| C | Intervention conseillee (all) | {fmt(len(intervention_conseillee_comment_context))} |',
    f'| C-worn-strong | Intervention conseillee — WORN_STRONG | {fmt(n_worn_strong)} |',
    f'| D | CRITIQUE + advisory-only comment | {fmt(len(critique_with_advisory_comments))} |',
    f'| E | CRITIQUE/IMMOBILISE + strong defect comment | {fmt(len(critical_comment_evidence))} |',
    '',
    '## Comment Evidence Level Distribution',
    '',
    '| Level | Count | % of all rows |',
    '|-------|-------|---------------|',
]

for level in EVIDENCE_ORDER:
    cnt = int(ev_counts.get(level, 0))
    pct = round(100 * cnt / n_total, 1)
    md_lines.append(f'| {level} | {fmt(cnt)} | {pct}% |')

md_lines += [
    '',
    '## Exported Files',
    '',
    '| File | Description |',
    '|------|-------------|',
    '| `staffim_comment_profile.csv` | Global counts and keyword coverage |',
    '| `comments_with_ok_value.csv` | Audit A — OK value + defect keyword |',
    '| `proposition_faite_comment_context.csv` | Audit B — PROPOSITION FAITE rows |',
    '| `intervention_conseillee_comment_context.csv` | Audit C — Intervention conseillee rows |',
    '| `critique_with_advisory_comments.csv` | Audit D — CRITIQUE + advisory comment |',
    '| `critical_comment_evidence.csv` | Audit E — CRITIQUE/IMMOBILISE + strong defect |',
    '| `staffim_comment_examples.csv` | 24 representative scenario examples |',
    '| `staffim_comment_crosstabs.csv` | Cross-tabs (5 dimensions x evidence level) |',
    '',
    '## Recommendation',
    '',
    'See notebook Section 13 for the final recommendation.',
    '',
    '---',
    '*Generated by `notebooks/04_staffim_comment_analysis_for_vhs.ipynb`*',
]

md_text = '\n'.join(md_lines)
md_path = OUTPUT_DIR / 'staffim_comment_analysis_summary.md'
md_path.write_text(md_text, encoding='utf-8')
print(f'Saved: {md_path}')
print()
print(md_text)

Saved: C:\Users\wiem\Downloads\Projet PFE\IRIS_AUTO_FRAUD\data\quality_reports\vhs\staffim_comment_analysis\staffim_comment_analysis_summary.md

# STAFFIM Comment Analysis — Summary Report

**Run ID:** `VHS_BALANCED_V2_20260630_133318`  
**Profile:** `VHS_BALANCED_V2`  
**Analysis date:** 2026-07-02  

---

## Dataset

| Metric | Value |
|--------|-------|
| Total checkpoint rows | 12,298 |
| Distinct inspections | 286 |
| Distinct vehicles | 280 |
| Rows with non-empty comment | 3,970 (32.3%) |

## Audit Findings

| Audit | Description | Rows |
|-------|-------------|------|
| A | OK/Bon value with defect keyword in comment | 2,119 |
| B | PROPOSITION FAITE (all) | 105 |
| B-advisory | PROPOSITION FAITE — advisory comment only | 2 |
| B-strong | PROPOSITION FAITE — strong defect in comment | 27 |
| C | Intervention conseillee (all) | 575 |
| C-worn-strong | Intervention conseillee — WORN_STRONG | 269 |
| D | CRITIQUE + advisory-only comment | 6 |
| E | CRITIQUE/IMMOBILISE + strong def

---
## 13. Final Recommendation

*This section is completed after reviewing the cross-tab and audit results produced above. Three options are defined; select the one whose criteria match the observed data.*

---

### Option A — Keep `VHS_BALANCED_V2` Unchanged

**Apply when all of the following hold:**
- Audit E (CRITIQUE/IMMOBILISE with strong defect comments) accounts for a large share of severe decisions — comments confirm the structured classification.
- Audit D (CRITIQUE with advisory-only comments) is small relative to Audit E.
- Cross-tab Section 8 shows that BROKEN / CRITIQUE rows carry predominantly `STRONG_DEFECT` or `NO_COMMENT` evidence, not `ADVISORY`.
- Audit B shows that PROPOSITION FAITE rows with advisory-only comments are few and do not materially drive CRITIQUE decisions.

**Conclusion:** Structured STAFFIM fields are reliable proxies for comment content. Comments add interpretive colour but reveal no systematic misclassification. `VHS_BALANCED_V2` remains the business reference version.

**Next step:** Store `comment_evidence_level` in a side table as audit support (Phase 2 of the roadmap) without modifying any score.

---

### Option B — Prepare `VHS_BALANCED_V3`

**Apply when:**
- Audit B shows a substantial proportion of `PROPOSITION FAITE` rows carry advisory-only comments (`n_prop_adv` >> `n_prop_strong`).
- These advisory PROPOSITION FAITE rows drive a measurable share of CRITIQUE decisions (Audit D count is significant).
- Cross-tab Section 8 shows a high `ADVISORY` share within the BROKEN or CRITIQUE rows, suggesting the current BROKEN classification is too broad.

**Proposed change for V3:**
Introduce a new observed status `ADVISORY_ONLY` for `PROPOSITION FAITE` rows where `est_anomalie_critique = false` and `tier` is not VITAL. Apply a reduced penalty (e.g. `penalty_worn * 0.5`) instead of the full `penalty_broken`.

**Prerequisite before any code change:**
Domain expert (technical inspection specialist) must validate Audit B findings and confirm that the advisory-comment sub-group is genuinely lower-severity.

---

### Option C — Keep Comments as Audit Evidence Only

**Apply when:**
- `NO_COMMENT` and `WEAK_OR_UNCLEAR` dominate across all decision categories, including CRITIQUE — comments are too sparse or vague to yield reliable signals.
- Keyword recall is visibly low (top-token analysis shows mostly non-safety terms).
- Audit B and Audit D counts are too small to justify a rule change.

**Conclusion:** Comments are too inconsistent to be safely scored. They should be preserved in the data model and surfaced in the inspection review UI for human analysts, but excluded from automated scoring indefinitely. Development effort should focus on improving `mart.dim_checkpoint` coverage or adding new structured STAFFIM fields.

---

> **How to apply:** Run all notebook cells, inspect the cross-tab compact summary (cell `12-crosstab-summary`) and the audit summary table (cell `10-audit-summary`), then select the option whose criteria best match the observed numbers. Record the chosen option and the key counts that motivated it in `docs/vhs/vhs_v3_decision.md`.

---
## Updated interpretation for ambiguous values

Recommendation for a future VHS version.
This table does **not** modify `VHS_BALANCED_V2`.
The current notebook is exploratory and read-only.

| `valeur_controle` | `est_anomalie` | `est_anomalie_critique` | `recommended_status` | Explanation |
|---|:---:|:---:|---|---|
| Bon | false | false | OK | Conformity confirmed |
| Contrôle OK | false | false | OK | Control validated |
| OUI | false | false | OK | Positive answer / compliant |
| Défectueux | true | true | BROKEN | Confirmed defect |
| Défectueux | true | false | BROKEN | Explicit defect wording remains strong even if critical flag is false |
| Contrôle non OK | true | true | BROKEN | Explicit non-conformity |
| Contrôle non OK | true | false | BROKEN | Explicit failed control |
| Intervention conseillée | true | false | WORN | Advisory intervention, non-critical |
| Intervention conseillée | true | true | WORN_STRONG | Strong advisory intervention, not necessarily broken |
| PROPOSITION FAITE | true | false | WORN | Proposed action, non-critical |
| PROPOSITION FAITE | true | true | WORN_STRONG | Proposed action with critical flag, not automatically broken |
| Proposition faite | true | false | WORN | Proposed action, non-critical |
| Proposition faite | true | true | WORN_STRONG | Proposed action with critical flag, not automatically broken |
| NON | true | false | WORN | Negative answer / non-critical non-conformity |
| NON | true | true | WORN_STRONG | Future-safe rule; critical negative answer but not explicit broken component |
| NON | false | false | UNKNOWN | Negative answer without anomaly flag; requires context |
| Réparation effectuée suite à l’accord client | any | any | REPAIRED | Repair completed; kept for traceability |
| Missing / not renseigné | false | false | UNKNOWN | Not enough information |

---

> **Note:** Although no case of `NON + est_anomalie_critique=true` was observed in the
> current dataset, this combination is included in the table to cover possible future data.
> It should be interpreted as **WORN_STRONG** rather than BROKEN, because NON expresses a
> negative answer or non-conformity, but does not explicitly confirm that a component is broken.

> **This table is a recommendation for a future VHS version. It does not modify
> VHS_BALANCED_V2 yet. The current notebook is exploratory and read-only.**

---

*Notebook: `notebooks/04_staffim_comment_analysis_for_vhs.ipynb`*  
*VHS run: `VHS_BALANCED_V2_20260630_133318`*  
*Analysis date: 2026-07-02*